# Model Experiment: Patch Time Series Transformer

PatchTST-inspired neural forecasting model for weekly Walmart Store-Dept sales.

The goal is to test whether a Transformer model over patched sales-history sequences can improve over simpler neural baselines such as DLinear and provide a useful comparison to the more feature-heavy TFT model.

We use point forecasting instead of probabilistic or quantile forecasting because the Kaggle metric is Weighted Mean Absolute Error (WMAE).

PatchTST is designed for long-horizon time-series forecasting by splitting the historical sequence into patches before applying a Transformer encoder.

Instead of treating each weekly observation as separate Transformer token, PatchTST groups consecutive weeks into local patches. For example with 52-week context and patch length 4 model sees roughly 13 patch tokens rather than 52 individual weekly tokens.

For our task:

- input context: previous 52 weeks
- prediction horizon: next 39 weeks
- output: 39 future Weekly_Sales predictions for each Store-Dept series

Unlike DLinear which uses mostly linear mappings from past target values, PatchTST learns nonlinear relationships between historical patches using self-attention.

Main experiment: target-only PatchTST

The initial PatchTST model uses:

- past target sequence: previous scaled Weekly_Sales values
- patching over the 52-week context window
- Transformer encoder over patch embeddings
- direct multi-step prediction of the next 39 weeks

Although the model is target-only, the dataset still keeps `IsHoliday` as known future field so that validation WMAE and optional holiday-weighted losses can be computed correctly.

Validation strategy:

- last-39 validation is used for the main PatchTST grid search because it provides many more training windows and was the best final selector for TFT
- calendar-aligned validation is used as a secondary realism check on the best last-39 configurations
- optional PatchTST-X with static and future covariate correction may be tested only if target-only PatchTST appears to miss important holiday or calendar effects

## Setup and Imports

In [1]:
from pathlib import Path
import sys
import os
import json
import random
from copy import deepcopy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt

In [2]:
# cwd = Path.cwd().resolve()
# repo_root = cwd if (cwd / "src").exists() else cwd.parent

# if str(repo_root) not in sys.path:
#     sys.path.insert(0, str(repo_root))

# ==========================================

# repo_root = Path("/content/drive/MyDrive/Machine Learning/Walmart").resolve()

# assert (repo_root / "src").exists(), f"src not found at {repo_root / 'src'}"

# print("Repo root:", repo_root)
# print("src exists:", (repo_root / "src").exists())
# print("data exists:", (repo_root / "data" / "raw").exists())

# os.chdir(repo_root)

# ==========================================

import shutil
import zipfile

KAGGLE_INPUT = Path("/kaggle/input")
repo_root = Path("/kaggle/working/Walmart").resolve()
data_raw_dir = repo_root / "data" / "raw"

repo_root.mkdir(parents=True, exist_ok=True)
data_raw_dir.mkdir(parents=True, exist_ok=True)

print("Available /kaggle/input folders:")
for p in KAGGLE_INPUT.iterdir():
    print(" -", p)


src_candidates = [
    p for p in KAGGLE_INPUT.rglob("src")
    if p.is_dir() and (p / "data").exists() and (p / "features").exists()
]

if not src_candidates:
    raise FileNotFoundError(
        "Could not find your project src/ folder under /kaggle/input. "
        "Make sure your Kaggle Dataset contains the src directory."
    )

source_src = src_candidates[0]
target_src = repo_root / "src"

if target_src.exists():
    shutil.rmtree(target_src)

shutil.copytree(source_src, target_src)

print("\nCopied src from:", source_src)
print("Copied src to:", target_src)

required_csvs = ["train.csv", "test.csv", "features.csv", "stores.csv"]

for csv_name in required_csvs:
    matches = list(KAGGLE_INPUT.rglob(csv_name))
    if matches:
        shutil.copy2(matches[0], data_raw_dir / csv_name)
        print(f"Copied {csv_name} from:", matches[0])

for zip_path in KAGGLE_INPUT.rglob("*.zip"):
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            names = z.namelist()
            wanted = [name for name in names if Path(name).name in required_csvs]

            for name in wanted:
                out_name = Path(name).name
                with z.open(name) as src_file, open(data_raw_dir / out_name, "wb") as dst_file:
                    shutil.copyfileobj(src_file, dst_file)

                print(f"Extracted {out_name} from:", zip_path)
    except zipfile.BadZipFile:
        pass

missing = [name for name in required_csvs if not (data_raw_dir / name).exists()]

if missing:
    print("\nFiles currently in data/raw:")
    for p in data_raw_dir.iterdir():
        print(" -", p.name)

    raise FileNotFoundError(f"Missing required raw files: {missing}")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.chdir(repo_root)

print("\nRepo root:", repo_root)
print("src exists:", (repo_root / "src").exists())
print("data exists:", data_raw_dir.exists())
print("Raw data files:", sorted(p.name for p in data_raw_dir.iterdir()))

from src.data import load_raw_data, last_n_weeks_split, calendar_aligned_split
from src.features import WalmartBasePreprocessor, WalmartNeuralPreprocessor
from src.datasets import (
    WalmartPrecomputedTrainingWindowDataset,
    WalmartPrecomputedForecastWindowDataset,
    FastTensorDataLoader,
)

Available /kaggle/input folders:
 - /kaggle/input/competitions
 - /kaggle/input/datasets

Copied src from: /kaggle/input/datasets/myvari/walmart-project-code/src
Copied src to: /kaggle/working/Walmart/src
Copied stores.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv
Extracted train.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip
Extracted features.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip
Extracted test.csv from: /kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip

Repo root: /kaggle/working/Walmart
src exists: True
data exists: True
Raw data files: ['features.csv', 'stores.csv', 'test.csv', 'train.csv']


In [3]:
DATA_DIR = repo_root / "data" / "raw"

CONTEXT_LENGTH = 52
PREDICTION_LENGTH = 39

BATCH_SIZE = 256
SEED = 42

MLFLOW_EXPERIMENT_NAME = "PatchTST_Training"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Repo root:", repo_root)
print("Data dir:", DATA_DIR)
print("Device:", DEVICE)

Repo root: /kaggle/working/Walmart
Data dir: /kaggle/working/Walmart/data/raw
Device: cuda


In [4]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## Dagshub/Mlflow initialization

In [5]:
pip install dagshub mlflow pandas matplotlib seaborn skops --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 87.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 88.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.3/121.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
import dagshub
import mlflow

dagshub.init(repo_owner='LukaBatilashvili07', repo_name='walmart-sales-forecasting', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=a244f7e4-bedc-404f-9dce-ba208602cc18&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=2c447baabef2a22468a3e3f454baef960f1e0dded2da5c6274a4f2a419cdc826




Accessing as myvari

Initialized MLflow to track repo "LukaBatilashvili07/walmart-sales-forecasting"

Repository LukaBatilashvili07/walmart-sales-forecasting initialized!

## Load Dataset and Time Split

In [7]:
train, test, stores, features = load_raw_data(DATA_DIR)

for df in [train, test, features]:
    df["Date"] = pd.to_datetime(df["Date"])

print("train:", train.shape)
print("test:", test.shape)
print("stores:", stores.shape)
print("features:", features.shape)

print("\nTrain date range:")
print(train["Date"].min(), "->", train["Date"].max())

print("\nTest date range:")
print(test["Date"].min(), "->", test["Date"].max())

display(train.head())
display(test.head())
display(stores.head())
display(features.head())

train: (421570, 5)
test: (115064, 4)
stores: (45, 3)
features: (8190, 12)

Train date range:
2010-02-05 00:00:00 -> 2012-10-26 00:00:00

Test date range:
2012-11-02 00:00:00 -> 2013-07-26 00:00:00


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [8]:
assert {"Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"}.issubset(train.columns)
assert {"Store", "Dept", "Date", "IsHoliday"}.issubset(test.columns)
assert {"Store", "Type", "Size"}.issubset(stores.columns)
assert {"Store", "Date", "IsHoliday"}.issubset(features.columns)

print("Raw data checks passed.")

Raw data checks passed.


In [9]:
# split A: last 39 weeks of train

train_raw_part, valid_raw_part = last_n_weeks_split(
    train,
    n_weeks=PREDICTION_LENGTH,
    date_col="Date",
)

print("Last-39 split")
print("train_raw_part:", train_raw_part.shape)
print("valid_raw_part:", valid_raw_part.shape)

print("\nTrain split date range:")
print(train_raw_part["Date"].min(), "->", train_raw_part["Date"].max())

print("\nValidation split date range:")
print(valid_raw_part["Date"].min(), "->", valid_raw_part["Date"].max())

print("\nUnique train dates:", train_raw_part["Date"].nunique())
print("Unique validation dates:", valid_raw_part["Date"].nunique())

assert train_raw_part["Date"].max() < valid_raw_part["Date"].min()
assert valid_raw_part["Date"].nunique() == PREDICTION_LENGTH

print("Last-39 split checks passed.")

Last-39 split
train_raw_part: (305982, 5)
valid_raw_part: (115588, 5)

Train split date range:
2010-02-05 00:00:00 -> 2012-01-27 00:00:00

Validation split date range:
2012-02-03 00:00:00 -> 2012-10-26 00:00:00

Unique train dates: 104
Unique validation dates: 39
Last-39 split checks passed.


In [10]:
# split B: calendar-aligned validation

calendar_train_raw_part, calendar_valid_raw_part = calendar_aligned_split(
    train,
    valid_start="2011-11-04",
    valid_end="2012-07-27",
    date_col="Date",
)

print("Calendar-aligned split")
print("calendar_train_raw_part:", calendar_train_raw_part.shape)
print("calendar_valid_raw_part:", calendar_valid_raw_part.shape)

print("\nCalendar train date range:")
print(calendar_train_raw_part["Date"].min(), "->", calendar_train_raw_part["Date"].max())

print("\nCalendar validation date range:")
print(calendar_valid_raw_part["Date"].min(), "->", calendar_valid_raw_part["Date"].max())

print("\nUnique calendar train dates:", calendar_train_raw_part["Date"].nunique())
print("Unique calendar validation dates:", calendar_valid_raw_part["Date"].nunique())

assert calendar_train_raw_part["Date"].max() < calendar_valid_raw_part["Date"].min()
assert calendar_valid_raw_part["Date"].nunique() == PREDICTION_LENGTH

print("Calendar-aligned split checks passed.")

Calendar-aligned split
calendar_train_raw_part: (267184, 5)
calendar_valid_raw_part: (115856, 5)

Calendar train date range:
2010-02-05 00:00:00 -> 2011-10-28 00:00:00

Calendar validation date range:
2011-11-04 00:00:00 -> 2012-07-27 00:00:00

Unique calendar train dates: 91
Unique calendar validation dates: 39
Calendar-aligned split checks passed.


## Base + neural preprocessing

In [11]:
base_preprocessor = WalmartBasePreprocessor()
base_preprocessor.fit(stores, features)

# last-39
last39_train_base = base_preprocessor.transform(train_raw_part)
last39_valid_base = base_preprocessor.transform(valid_raw_part)

# calendar-aligned
calendar_train_base = base_preprocessor.transform(calendar_train_raw_part)
calendar_valid_base = base_preprocessor.transform(calendar_valid_raw_part)

print("Last-39 base:")
print("last39_train_base:", last39_train_base.shape)
print("last39_valid_base:", last39_valid_base.shape)

print("\nCalendar base:")
print("calendar_train_base:", calendar_train_base.shape)
print("calendar_valid_base:", calendar_valid_base.shape)

assert len(last39_train_base) == len(train_raw_part)
assert len(last39_valid_base) == len(valid_raw_part)

assert len(calendar_train_base) == len(calendar_train_raw_part)
assert len(calendar_valid_base) == len(calendar_valid_raw_part)

display(last39_train_base.head())

Last-39 base:
last39_train_base: (305982, 42)
last39_valid_base: (115588, 42)

Calendar base:
calendar_train_base: (267184, 42)
calendar_valid_base: (115856, 42)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,...,markdown_available_period,Year,Month,WeekOfYear,Week_sin,Week_cos,IsSuperBowl,IsLaborDay,IsThanksgiving,IsChristmas
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.0,...,0,2010,2,5,0.568065,0.822984,0,0,0,0
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,0.0,...,0,2010,2,6,0.663123,0.748511,1,0,0,0
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,0.0,...,0,2010,2,7,0.748511,0.663123,0,0,0,0
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,0.0,...,0,2010,2,8,0.822984,0.568065,0,0,0,0
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,0.0,...,0,2010,3,9,0.885456,0.464723,0,0,0,0


In [12]:
required_base_cols = [
    "Store", "Dept", "Date", "Weekly_Sales",
    "Type", "Size",
    "IsHoliday",
    "Temperature", "Fuel_Price", "CPI", "Unemployment",
    "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5",
    "MarkDown1_was_missing", "MarkDown2_was_missing", "MarkDown3_was_missing",
    "MarkDown4_was_missing", "MarkDown5_was_missing",
    "total_markdown", "abs_total_markdown",
    "positive_markdown_sum", "negative_markdown_sum",
    "markdown_missing_count", "markdown_available_period",
    "has_markdown_signal",
    "Week_sin", "Week_cos",
    "IsSuperBowl", "IsLaborDay", "IsThanksgiving", "IsChristmas",
]

base_panels = {
    "last39_train_base": last39_train_base,
    "last39_valid_base": last39_valid_base,
    "calendar_train_base": calendar_train_base,
    "calendar_valid_base": calendar_valid_base,
}

for name, df in base_panels.items():
    missing = [c for c in required_base_cols if c not in df.columns]
    print(name, "missing:", missing)
    assert not missing, f"{name} is missing columns: {missing}"

print("Base preprocessing checks passed.")

last39_train_base missing: []
last39_valid_base missing: []
calendar_train_base missing: []
calendar_valid_base missing: []
Base preprocessing checks passed.


In [13]:
# last-39 neural preprocessing
last39_neural_preprocessor = WalmartNeuralPreprocessor()
last39_neural_preprocessor.fit(last39_train_base)

last39_train_panel = last39_neural_preprocessor.transform(last39_train_base)
last39_valid_panel = last39_neural_preprocessor.transform(last39_valid_base)

last39_dataset_cols = last39_neural_preprocessor.get_dataset_columns()

# calendar-aligned neural preprocessing
calendar_neural_preprocessor = WalmartNeuralPreprocessor()
calendar_neural_preprocessor.fit(calendar_train_base)

calendar_train_panel = calendar_neural_preprocessor.transform(calendar_train_base)
calendar_valid_panel = calendar_neural_preprocessor.transform(calendar_valid_base)

calendar_dataset_cols = calendar_neural_preprocessor.get_dataset_columns()

print("Last-39 panels:")
print("last39_train_panel:", last39_train_panel.shape)
print("last39_valid_panel:", last39_valid_panel.shape)

print("\nCalendar panels:")
print("calendar_train_panel:", calendar_train_panel.shape)
print("calendar_valid_panel:", calendar_valid_panel.shape)

print("\nDataset columns:")
for key, value in calendar_dataset_cols.items():
    print(f"{key}: {value}")

Last-39 panels:
last39_train_panel: (305982, 66)
last39_valid_panel: (115588, 66)

Calendar panels:
calendar_train_panel: (267184, 66)
calendar_valid_panel: (115856, 66)

Dataset columns:
target_col: Weekly_Sales_scaled
series_col: series_id
static_cat_cols: ['Store_id', 'Dept_id', 'Type_id']
static_real_cols: ['Size_scaled']
known_future_real_cols: ['Temperature_scaled', 'Fuel_Price_scaled', 'CPI_scaled', 'Unemployment_scaled', 'MarkDown1_scaled', 'MarkDown2_scaled', 'MarkDown3_scaled', 'MarkDown4_scaled', 'MarkDown5_scaled', 'total_markdown_scaled', 'abs_total_markdown_scaled', 'positive_markdown_sum_scaled', 'negative_markdown_sum_scaled', 'markdown_missing_count_scaled', 'Week_sin_scaled', 'Week_cos_scaled', 'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas', 'has_markdown_signal', 'markdown_available_period', 'MarkDown1_was_missing', 'MarkDown2_was_missing', 'MarkDown3_was_missing', 'MarkDown4_was_missing', 'MarkDown5_was_missing']


In [14]:
expected_neural_cols = [
    "series_id",
    "Store_id", "Dept_id", "Type_id",
    "Weekly_Sales_scaled",
    "target_mean", "target_std",
]

neural_panels = {
    "last39_train_panel": last39_train_panel,
    "last39_valid_panel": last39_valid_panel,
    "calendar_train_panel": calendar_train_panel,
    "calendar_valid_panel": calendar_valid_panel,
}

for name, df in neural_panels.items():
    for col in expected_neural_cols:
        assert col in df.columns, f"Missing from {name}: {col}"

    assert df["Weekly_Sales_scaled"].notna().all(), f"NaN target scale in {name}"
    assert df["target_mean"].notna().all(), f"NaN target_mean in {name}"
    assert df["target_std"].notna().all(), f"NaN target_std in {name}"
    assert (df["target_std"] > 0).all(), f"Non-positive target_std in {name}"

print("Neural preprocessing checks passed.")

Neural preprocessing checks passed.


In [49]:
print("Dataset column definitions from neural preprocessor")

print("\nTarget column:")
print(calendar_dataset_cols["target_col"])

print("\nSeries column:")
print(calendar_dataset_cols["series_col"])

print("\nAvailable static categorical columns:")
print(calendar_dataset_cols["static_cat_cols"])

print("\nAvailable static real columns:")
print(calendar_dataset_cols["static_real_cols"])

print("\nAvailable known future/time-varying feature columns:")
print("Count:", len(calendar_dataset_cols["known_future_real_cols"]))
print(calendar_dataset_cols["known_future_real_cols"])

assert last39_dataset_cols["target_col"] == calendar_dataset_cols["target_col"]
assert last39_dataset_cols["series_col"] == calendar_dataset_cols["series_col"]
assert last39_dataset_cols["known_future_real_cols"] == calendar_dataset_cols["known_future_real_cols"]
assert last39_dataset_cols["static_cat_cols"] == calendar_dataset_cols["static_cat_cols"]
assert last39_dataset_cols["static_real_cols"] == calendar_dataset_cols["static_real_cols"]

assert "IsHoliday" in calendar_dataset_cols["known_future_real_cols"]

Dataset column definitions from neural preprocessor

Target column:
Weekly_Sales_scaled

Series column:
series_id

Available static categorical columns:
['Store_id', 'Dept_id', 'Type_id']

Available static real columns:
['Size_scaled']

Available known future/time-varying feature columns:
Count: 28
['Temperature_scaled', 'Fuel_Price_scaled', 'CPI_scaled', 'Unemployment_scaled', 'MarkDown1_scaled', 'MarkDown2_scaled', 'MarkDown3_scaled', 'MarkDown4_scaled', 'MarkDown5_scaled', 'total_markdown_scaled', 'abs_total_markdown_scaled', 'positive_markdown_sum_scaled', 'negative_markdown_sum_scaled', 'markdown_missing_count_scaled', 'Week_sin_scaled', 'Week_cos_scaled', 'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas', 'has_markdown_signal', 'markdown_available_period', 'MarkDown1_was_missing', 'MarkDown2_was_missing', 'MarkDown3_was_missing', 'MarkDown4_was_missing', 'MarkDown5_was_missing']


##  Dataset and DataLoaders

In [16]:
def remove_markdown_features(cols: list[str]) -> list[str]:
    return [
        col for col in cols
        if "markdown" not in col.lower()
    ]


def make_patchtst_target_only_cols(dataset_cols: dict) -> dict:
    """
    Target-only PatchTST uses only past_target as model input.

    IsHoliday is still kept in known_future_real_cols so that WMAE and
    optional holiday-weighted losses can be computed from the batch.
    """
    return {
        "target_col": dataset_cols["target_col"],
        "series_col": dataset_cols["series_col"],
        "static_cat_cols": [],
        "static_real_cols": [],
        "known_future_real_cols": ["IsHoliday"],
    }


def make_patchtst_x_cols(dataset_cols: dict) -> dict:
    """
    Optional later PatchTST-X feature set.

    This is not used in the first target-only experiment but it is defined
    here so the notebook can be extended cleanly if needed.
    """
    known_future_cols = remove_markdown_features(
        dataset_cols["known_future_real_cols"]
    )

    return {
        "target_col": dataset_cols["target_col"],
        "series_col": dataset_cols["series_col"],
        "static_cat_cols": dataset_cols["static_cat_cols"],
        "static_real_cols": dataset_cols["static_real_cols"],
        "known_future_real_cols": known_future_cols,
    }


patchtst_calendar_target_cols = make_patchtst_target_only_cols(calendar_dataset_cols)
patchtst_last39_target_cols = make_patchtst_target_only_cols(last39_dataset_cols)

patchtst_calendar_x_cols = make_patchtst_x_cols(calendar_dataset_cols)
patchtst_last39_x_cols = make_patchtst_x_cols(last39_dataset_cols)

print("PatchTST target-only calendar columns:")
print(patchtst_calendar_target_cols)

print("\nPatchTST target-only last-39 columns:")
print(patchtst_last39_target_cols)

print("\PatchTST-X known future features:", len(patchtst_calendar_x_cols["known_future_real_cols"]))
print(patchtst_calendar_x_cols["known_future_real_cols"])

PatchTST target-only calendar columns:
{'target_col': 'Weekly_Sales_scaled', 'series_col': 'series_id', 'static_cat_cols': [], 'static_real_cols': [], 'known_future_real_cols': ['IsHoliday']}

PatchTST target-only last-39 columns:
{'target_col': 'Weekly_Sales_scaled', 'series_col': 'series_id', 'static_cat_cols': [], 'static_real_cols': [], 'known_future_real_cols': ['IsHoliday']}
\PatchTST-X known future features: 11
['Temperature_scaled', 'Fuel_Price_scaled', 'CPI_scaled', 'Unemployment_scaled', 'Week_sin_scaled', 'Week_cos_scaled', 'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas']


<>:56: SyntaxWarning: invalid escape sequence '\P'
<>:56: SyntaxWarning: invalid escape sequence '\P'
/tmp/ipykernel_58/4217330780.py:56: SyntaxWarning: invalid escape sequence '\P'
  print("\PatchTST-X known future features:", len(patchtst_calendar_x_cols["known_future_real_cols"]))


In [17]:
def make_full_horizon_validation_panel(
    valid_panel: pd.DataFrame,
    prediction_length: int,
    series_col: str = "series_id",
) -> tuple[pd.DataFrame, pd.Index]:
    group_sizes = valid_panel.groupby(series_col).size()

    full_horizon_series = group_sizes[
        group_sizes == prediction_length
    ].index

    valid_panel_full = valid_panel[
        valid_panel[series_col].isin(full_horizon_series)
    ].copy()

    assert len(valid_panel_full) == len(full_horizon_series) * prediction_length

    return valid_panel_full, full_horizon_series

In [18]:
calendar_valid_panel_full, calendar_full_horizon_series = make_full_horizon_validation_panel(
    calendar_valid_panel,
    prediction_length=PREDICTION_LENGTH,
    series_col=patchtst_calendar_target_cols["series_col"],
)

last39_valid_panel_full, last39_full_horizon_series = make_full_horizon_validation_panel(
    last39_valid_panel,
    prediction_length=PREDICTION_LENGTH,
    series_col=patchtst_last39_target_cols["series_col"],
)

print("Calendar validation:")
print("total series:", calendar_valid_panel["series_id"].nunique())
print("full-horizon series:", len(calendar_full_horizon_series))
print("calendar_valid_panel rows:", len(calendar_valid_panel))
print("calendar_valid_panel_full rows:", len(calendar_valid_panel_full))

print("\nLast-39 validation:")
print("total series:", last39_valid_panel["series_id"].nunique())
print("full-horizon series:", len(last39_full_horizon_series))
print("last39_valid_panel rows:", len(last39_valid_panel))
print("last39_valid_panel_full rows:", len(last39_valid_panel_full))

Calendar validation:
total series: 3233
full-horizon series: 2762
calendar_valid_panel rows: 115856
calendar_valid_panel_full rows: 107718

Last-39 validation:
total series: 3204
full-horizon series: 2762
last39_valid_panel rows: 115588
last39_valid_panel_full rows: 107718


In [19]:
def build_patchtst_data_bundle(
    train_panel: pd.DataFrame,
    valid_panel_full: pd.DataFrame,
    cols: dict,
    batch_size: int = BATCH_SIZE,
) -> dict:
    train_dataset = WalmartPrecomputedTrainingWindowDataset(
        train_panel,
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        target_col=cols["target_col"],
        series_col=cols["series_col"],
        static_cat_cols=cols["static_cat_cols"],
        static_real_cols=cols["static_real_cols"],
        known_future_real_cols=cols["known_future_real_cols"],
    )

    valid_dataset = WalmartPrecomputedForecastWindowDataset(
        history_df=train_panel,
        future_df=valid_panel_full,
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        target_col=cols["target_col"],
        series_col=cols["series_col"],
        static_cat_cols=cols["static_cat_cols"],
        static_real_cols=cols["static_real_cols"],
        known_future_real_cols=cols["known_future_real_cols"],
    )

    train_loader = FastTensorDataLoader(
        train_dataset.tensors,
        batch_size=batch_size,
        shuffle=True,
    )

    valid_loader = FastTensorDataLoader(
        valid_dataset.tensors,
        batch_size=batch_size,
        shuffle=False,
    )

    static_cat_cardinalities = [
        int(max(train_panel[col].max(), valid_panel_full[col].max())) + 1
        for col in cols["static_cat_cols"]
    ]

    num_static_reals = len(cols["static_real_cols"])
    num_known_reals = len(cols["known_future_real_cols"])

    assert "IsHoliday" in cols["known_future_real_cols"]
    holiday_feature_idx = cols["known_future_real_cols"].index("IsHoliday")

    assert train_dataset.tensors["past_target"].shape[1] == CONTEXT_LENGTH
    assert train_dataset.tensors["future_target"].shape[1] == PREDICTION_LENGTH

    assert valid_dataset.tensors["past_target"].shape[1] == CONTEXT_LENGTH
    assert valid_dataset.tensors["future_target"].shape[1] == PREDICTION_LENGTH

    assert train_dataset.tensors["past_known_reals"].shape[2] == num_known_reals
    assert train_dataset.tensors["future_known_reals"].shape[2] == num_known_reals
    assert valid_dataset.tensors["past_known_reals"].shape[2] == num_known_reals
    assert valid_dataset.tensors["future_known_reals"].shape[2] == num_known_reals

    assert train_dataset.tensors["static_categoricals"].shape[1] == len(cols["static_cat_cols"])
    assert valid_dataset.tensors["static_categoricals"].shape[1] == len(cols["static_cat_cols"])

    assert train_dataset.tensors["static_reals"].shape[1] == num_static_reals
    assert valid_dataset.tensors["static_reals"].shape[1] == num_static_reals

    return {
        "cols": cols,
        "train_dataset": train_dataset,
        "valid_dataset": valid_dataset,
        "train_loader": train_loader,
        "valid_loader": valid_loader,
        "static_cat_cardinalities": static_cat_cardinalities,
        "num_static_reals": num_static_reals,
        "num_known_reals": num_known_reals,
        "holiday_feature_idx": holiday_feature_idx,
        "batch_size": batch_size,
    }

In [20]:
patchtst_calendar_target_data = build_patchtst_data_bundle(
    train_panel=calendar_train_panel,
    valid_panel_full=calendar_valid_panel_full,
    cols=patchtst_calendar_target_cols,
    batch_size=BATCH_SIZE,
)

patchtst_last39_target_data = build_patchtst_data_bundle(
    train_panel=last39_train_panel,
    valid_panel_full=last39_valid_panel_full,
    cols=patchtst_last39_target_cols,
    batch_size=BATCH_SIZE,
)

print("Calendar PatchTST target-only data:")
print("train windows:", len(patchtst_calendar_target_data["train_dataset"]))
print("valid forecast series:", len(patchtst_calendar_target_data["valid_dataset"]))
print("train batches:", len(patchtst_calendar_target_data["train_loader"]))
print("valid batches:", len(patchtst_calendar_target_data["valid_loader"]))
print("num known reals:", patchtst_calendar_target_data["num_known_reals"])
print("num static cats:", len(patchtst_calendar_target_data["static_cat_cardinalities"]))
print("num static reals:", patchtst_calendar_target_data["num_static_reals"])
print("holiday idx:", patchtst_calendar_target_data["holiday_feature_idx"])

print("\nLast-39 PatchTST target-only data:")
print("train windows:", len(patchtst_last39_target_data["train_dataset"]))
print("valid forecast series:", len(patchtst_last39_target_data["valid_dataset"]))
print("train batches:", len(patchtst_last39_target_data["train_loader"]))
print("valid batches:", len(patchtst_last39_target_data["valid_loader"]))
print("num known reals:", patchtst_last39_target_data["num_known_reals"])
print("num static cats:", len(patchtst_last39_target_data["static_cat_cardinalities"]))
print("num static reals:", patchtst_last39_target_data["num_static_reals"])
print("holiday idx:", patchtst_last39_target_data["holiday_feature_idx"])

Calendar PatchTST target-only data:
train windows: 2683
valid forecast series: 2762
train batches: 11
valid batches: 11
num known reals: 1
num static cats: 0
num static reals: 0
holiday idx: 0

Last-39 PatchTST target-only data:
train windows: 38464
valid forecast series: 2762
train batches: 151
valid batches: 11
num known reals: 1
num static cats: 0
num static reals: 0
holiday idx: 0


In [21]:
calendar_batch = next(iter(patchtst_calendar_target_data["train_loader"]))

print("Calendar PatchTST target-only train batch:")
for key, value in calendar_batch.items():
    print(key, tuple(value.shape), value.dtype)

valid_calendar_batch = next(iter(patchtst_calendar_target_data["valid_loader"]))

print("\nCalendar PatchTST target-only validation batch:")
for key, value in valid_calendar_batch.items():
    print(key, tuple(value.shape), value.dtype)

print("\nMain model input shape:")
print("past_target:", tuple(calendar_batch["past_target"].shape))

print("\nFields kept for evaluation/scaling:")
print("future_target:", tuple(calendar_batch["future_target"].shape))
print("future_known_reals:", tuple(calendar_batch["future_known_reals"].shape))
print("target_mean:", tuple(calendar_batch["target_mean"].shape))
print("target_std:", tuple(calendar_batch["target_std"].shape))

Calendar PatchTST target-only train batch:
past_target (256, 52) torch.float32
future_target (256, 39) torch.float32
past_known_reals (256, 52, 1) torch.float32
future_known_reals (256, 39, 1) torch.float32
static_categoricals (256, 0) torch.int64
static_reals (256, 0) torch.float32
target_mean (256,) torch.float32
target_std (256,) torch.float32

Calendar PatchTST target-only validation batch:
past_target (256, 52) torch.float32
future_target (256, 39) torch.float32
past_known_reals (256, 52, 1) torch.float32
future_known_reals (256, 39, 1) torch.float32
static_categoricals (256, 0) torch.int64
static_reals (256, 0) torch.float32
target_mean (256,) torch.float32
target_std (256,) torch.float32
store (256,) torch.int64
dept (256,) torch.int64

Main model input shape:
past_target: (256, 52)

Fields kept for evaluation/scaling:
future_target: (256, 39)
future_known_reals: (256, 39, 1)
target_mean: (256,)
target_std: (256,)


In [38]:
patchtst_calendar_x_data = build_patchtst_data_bundle(
    train_panel=calendar_train_panel,
    valid_panel_full=calendar_valid_panel_full,
    cols=patchtst_calendar_x_cols,
    batch_size=BATCH_SIZE,
)

patchtst_last39_x_data = build_patchtst_data_bundle(
    train_panel=last39_train_panel,
    valid_panel_full=last39_valid_panel_full,
    cols=patchtst_last39_x_cols,
    batch_size=BATCH_SIZE,
)

print("Calendar PatchTST-X data:")
print("train windows:", len(patchtst_calendar_x_data["train_dataset"]))
print("valid forecast series:", len(patchtst_calendar_x_data["valid_dataset"]))
print("train batches:", len(patchtst_calendar_x_data["train_loader"]))
print("valid batches:", len(patchtst_calendar_x_data["valid_loader"]))
print("static cat cardinalities:", patchtst_calendar_x_data["static_cat_cardinalities"])
print("num known reals:", patchtst_calendar_x_data["num_known_reals"])
print("num static reals:", patchtst_calendar_x_data["num_static_reals"])
print("holiday idx:", patchtst_calendar_x_data["holiday_feature_idx"])

print("\nLast-39 PatchTST-X data:")
print("train windows:", len(patchtst_last39_x_data["train_dataset"]))
print("valid forecast series:", len(patchtst_last39_x_data["valid_dataset"]))
print("train batches:", len(patchtst_last39_x_data["train_loader"]))
print("valid batches:", len(patchtst_last39_x_data["valid_loader"]))
print("static cat cardinalities:", patchtst_last39_x_data["static_cat_cardinalities"])
print("num known reals:", patchtst_last39_x_data["num_known_reals"])
print("num static reals:", patchtst_last39_x_data["num_static_reals"])
print("holiday idx:", patchtst_last39_x_data["holiday_feature_idx"])

Calendar PatchTST-X data:
train windows: 2683
valid forecast series: 2762
train batches: 11
valid batches: 11
static cat cardinalities: [46, 82, 4]
num known reals: 11
num static reals: 1
holiday idx: 6

Last-39 PatchTST-X data:
train windows: 38464
valid forecast series: 2762
train batches: 151
valid batches: 11
static cat cardinalities: [46, 82, 4]
num known reals: 11
num static reals: 1
holiday idx: 6


In [39]:
x_batch = next(iter(patchtst_last39_x_data["train_loader"]))

print("PatchTST-X train batch:")
for key, value in x_batch.items():
    print(key, tuple(value.shape), value.dtype)

PatchTST-X train batch:
past_target (256, 52) torch.float32
future_target (256, 39) torch.float32
past_known_reals (256, 52, 11) torch.float32
future_known_reals (256, 39, 11) torch.float32
static_categoricals (256, 3) torch.int64
static_reals (256, 1) torch.float32
target_mean (256,) torch.float32
target_std (256,) torch.float32


In [22]:
import time

def time_loader(loader, name="loader", max_batches=None):
    t0 = time.perf_counter()
    n_batches = 0
    n_samples = 0

    for batch in loader:
        n_batches += 1
        n_samples += batch["past_target"].shape[0]

        if max_batches is not None and n_batches >= max_batches:
            break

    elapsed = time.perf_counter() - t0

    print(f"{name}: {elapsed:.2f}s, batches={n_batches}, samples={n_samples}")
    print(f"seconds/batch: {elapsed / max(n_batches, 1):.4f}")


time_loader(patchtst_calendar_target_data["train_loader"], "PatchTST calendar target-only train loader")
time_loader(patchtst_calendar_target_data["valid_loader"], "PatchTST calendar target-only valid loader")
time_loader(patchtst_last39_target_data["train_loader"], "PatchTST last-39 target-only train loader")
time_loader(patchtst_last39_target_data["valid_loader"], "PatchTST last-39 target-only valid loader")

PatchTST calendar target-only train loader: 0.00s, batches=11, samples=2683
seconds/batch: 0.0003
PatchTST calendar target-only valid loader: 0.00s, batches=11, samples=2762
seconds/batch: 0.0002
PatchTST last-39 target-only train loader: 0.03s, batches=151, samples=38464
seconds/batch: 0.0002
PatchTST last-39 target-only valid loader: 0.00s, batches=11, samples=2762
seconds/batch: 0.0001


In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="PatchTST_Preprocessing") as run:
    mlflow.log_param("model_family", "PatchTST")
    mlflow.log_param("model_variant", "target_only")
    mlflow.log_param("context_length", CONTEXT_LENGTH)
    mlflow.log_param("prediction_length", PREDICTION_LENGTH)
    mlflow.log_param("base_preprocessor", "WalmartBasePreprocessor")
    mlflow.log_param("neural_preprocessor", "WalmartNeuralPreprocessor")
    mlflow.log_param("target_scaling", "series_mean_std_with_dept_store_global_fallback")
    mlflow.log_param("primary_validation_strategy", "last_39_weeks")
    mlflow.log_param("secondary_validation_strategy", "calendar_aligned_39_weeks")
    mlflow.log_param("feature_set", "target_only_with_IsHoliday_for_metric")
    mlflow.log_param("forecast_validation", "full_horizon_series_only")

    mlflow.log_metric("raw_train_rows", len(train))
    mlflow.log_metric("raw_test_rows", len(test))

    # last-39 split metrics
    mlflow.log_metric("last39_train_rows", len(train_raw_part))
    mlflow.log_metric("last39_valid_rows", len(valid_raw_part))
    mlflow.log_metric("last39_train_panel_rows", len(last39_train_panel))
    mlflow.log_metric("last39_valid_panel_rows", len(last39_valid_panel))
    mlflow.log_metric("last39_full_horizon_valid_series", len(patchtst_last39_target_data["valid_dataset"]))
    mlflow.log_metric("last39_train_windows", len(patchtst_last39_target_data["train_dataset"]))

    # calendar split metrics
    mlflow.log_metric("calendar_train_rows", len(calendar_train_raw_part))
    mlflow.log_metric("calendar_valid_rows", len(calendar_valid_raw_part))
    mlflow.log_metric("calendar_train_panel_rows", len(calendar_train_panel))
    mlflow.log_metric("calendar_valid_panel_rows", len(calendar_valid_panel))
    mlflow.log_metric("calendar_full_horizon_valid_series", len(patchtst_calendar_target_data["valid_dataset"]))
    mlflow.log_metric("calendar_train_windows", len(patchtst_calendar_target_data["train_dataset"]))

    # feature dimensions for target-only setup
    mlflow.log_metric("target_only_num_static_cat_cols", len(patchtst_calendar_target_cols["static_cat_cols"]))
    mlflow.log_metric("target_only_num_static_real_cols", len(patchtst_calendar_target_cols["static_real_cols"]))
    mlflow.log_metric("target_only_num_known_future_reals", len(patchtst_calendar_target_cols["known_future_real_cols"]))

    preprocessing_config = {
        "model_family": "PatchTST",
        "model_variant": "target_only",
        "context_length": CONTEXT_LENGTH,
        "prediction_length": PREDICTION_LENGTH,
        "primary_validation_strategy": "last_39_weeks",
        "secondary_validation_strategy": "calendar_aligned_39_weeks",
        "feature_set": "target_only_with_IsHoliday_for_metric",
        "patchtst_calendar_target_cols": patchtst_calendar_target_cols,
        "patchtst_last39_target_cols": patchtst_last39_target_cols,
        "optional_patchtst_calendar_x_cols": patchtst_calendar_x_cols,
        "optional_patchtst_last39_x_cols": patchtst_last39_x_cols,
        "required_base_cols": required_base_cols,
    }

    config_path = repo_root / "patchtst_preprocessing_config.json"

    with open(config_path, "w") as f:
        json.dump(preprocessing_config, f, indent=2)

    mlflow.log_artifact(str(config_path), artifact_path="config")
    config_path.unlink()

    patchtst_preprocessing_run_id = run.info.run_id

print("Logged PatchTST preprocessing run:", patchtst_preprocessing_run_id)

## Evaluation utilities

In [23]:
def weighted_mae_np(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    is_holiday = np.asarray(is_holiday).reshape(-1).astype(bool)

    weights = np.where(is_holiday, 5.0, 1.0)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


def inverse_scale_torch(y_scaled, target_mean, target_std):
    """
    y_scaled: [B, H]
    target_mean: [B]
    target_std: [B]
    """
    return y_scaled * target_std.unsqueeze(1) + target_mean.unsqueeze(1)


def move_batch_to_device(batch, device):
    return {
        key: value.to(device) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def make_loss_fn(loss_name: str):
    loss_name = loss_name.lower()

    if loss_name == "mse":
        return nn.MSELoss()

    if loss_name in ["mae", "l1"]:
        return nn.L1Loss()

    if loss_name == "huber":
        return nn.HuberLoss(delta=1.0)

    raise ValueError(f"Unknown loss_name: {loss_name}")

## PatchTST Model Definitions

In [24]:
class PatchTSTPointForecaster(nn.Module):
    """
    Target-only PatchTST-style point forecaster.

    Inputs:
        past_target: [B, context_length]

    Output:
        forecast: [B, prediction_length]

    Main idea:
        1. Split the past target sequence into patches.
        2. Project each patch into a d_model vector.
        3. Apply Transformer encoder self-attention over patch embeddings.
        4. Flatten encoded patches and predict the full forecast horizon.
    """
    def __init__(
        self,
        context_length: int,
        prediction_length: int,
        patch_length: int = 4,
        stride: int = 4,
        d_model: int = 64,
        num_layers: int = 2,
        attention_heads: int = 4,
        dim_feedforward: int | None = None,
        dropout: float = 0.1,
        pad_end: bool = True,
    ):
        super().__init__()

        if d_model % attention_heads != 0:
            raise ValueError(
                f"d_model={d_model} must be divisible by "
                f"attention_heads={attention_heads}"
            )

        if patch_length <= 0:
            raise ValueError("patch_length must be positive.")

        if stride <= 0:
            raise ValueError("stride must be positive.")

        if patch_length > context_length:
            raise ValueError(
                f"patch_length={patch_length} cannot be larger than "
                f"context_length={context_length}."
            )

        self.context_length = context_length
        self.prediction_length = prediction_length
        self.patch_length = patch_length
        self.stride = stride
        self.d_model = d_model
        self.pad_end = pad_end

        if pad_end:
            remainder = (context_length - patch_length) % stride
            self.padding = 0 if remainder == 0 else stride - remainder
        else:
            self.padding = 0

        padded_length = context_length + self.padding
        self.num_patches = ((padded_length - patch_length) // stride) + 1

        if dim_feedforward is None:
            dim_feedforward = 4 * d_model

        self.patch_projection = nn.Linear(patch_length, d_model)

        self.position_embedding = nn.Parameter(
            torch.zeros(1, self.num_patches, d_model)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=attention_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=False,
        )

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

        self.forecast_head = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Dropout(dropout),
            nn.Linear(self.num_patches * d_model, prediction_length),
        )

    def _make_patches(self, past_target):
        """
        past_target: [B, context_length]
        returns: [B, num_patches, patch_length]
        """
        if self.padding > 0:
            # extend instead of just 0
            last_value_padding = past_target[:, -1:].expand(
                -1,
                self.padding,
            )

            past_target = torch.cat(
                [past_target, last_value_padding],
                dim=1,
            )

        patches = past_target.unfold(
            dimension=1,
            size=self.patch_length,
            step=self.stride,
        )

        return patches

    def forward(
        self,
        past_target,
        past_known_reals=None,
        future_known_reals=None,
        static_categoricals=None,
        static_reals=None,
    ):
        # [B, num_patches, patch_length]
        patches = self._make_patches(past_target)

        # [B, num_patches, d_model]
        x = self.patch_projection(patches)

        x = x + self.position_embedding[:, :x.shape[1], :]
        x = self.dropout(x)

        # self-attention over patch embeddings
        x = self.encoder(x)
        x = self.norm(x)

        # direct multi-step forecast
        forecast = self.forecast_head(x)

        return forecast

In [25]:
test_batch = next(iter(patchtst_last39_target_data["train_loader"]))
test_batch = move_batch_to_device(test_batch, DEVICE)

patchtst_test_model = PatchTSTPointForecaster(
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    patch_length=4,
    stride=4,
    d_model=64,
    num_layers=2,
    attention_heads=4,
    dropout=0.1,
).to(DEVICE)

with torch.no_grad():
    test_output = patchtst_test_model(
        past_target=test_batch["past_target"],
    )

print("PatchTST output shape:", tuple(test_output.shape))
print("Expected shape:", tuple(test_batch["future_target"].shape))

assert test_output.shape == test_batch["future_target"].shape

print("Number of patches:", patchtst_test_model.num_patches)
print("Trainable parameters:", count_parameters(patchtst_test_model))

del patchtst_test_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("PatchTST shape test passed.")

PatchTST output shape: (256, 39)
Expected shape: (256, 39)
Number of patches: 13
Trainable parameters: 133735
PatchTST shape test passed.


In [40]:
class StaticFeatureEncoder(nn.Module):
    """
    Encodes static categorical and static real features into one vector per series.

    Inputs:
        static_categoricals: [B, C]
        static_reals:        [B, S]

    Output:
        static_context:      [B, hidden_dim]
    """
    def __init__(
        self,
        static_cat_cardinalities: list[int],
        num_static_reals: int,
        embedding_dim: int,
        hidden_dim: int,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.static_cat_cardinalities = static_cat_cardinalities
        self.num_static_reals = num_static_reals

        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, embedding_dim)
            for cardinality in static_cat_cardinalities
        ])

        input_dim = len(static_cat_cardinalities) * embedding_dim + num_static_reals

        if input_dim <= 0:
            raise ValueError(
                "PatchTST-X requires at least one static categorical or static real feature."
            )

        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(), # since we're closer to tranformer
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
        )

    def forward(self, static_categoricals, static_reals):
        embedded_parts = []

        for i, embedding in enumerate(self.embeddings):
            cat_values = static_categoricals[:, i].long()
            cat_values = cat_values.clamp(0, embedding.num_embeddings - 1)
            embedded_parts.append(embedding(cat_values))

        if embedded_parts:
            cat_context = torch.cat(embedded_parts, dim=1)
        else:
            cat_context = static_reals.new_zeros(static_reals.shape[0], 0)

        if self.num_static_reals > 0:
            static_input = torch.cat([cat_context, static_reals], dim=1)
        else:
            static_input = cat_context

        return self.projection(static_input)

In [44]:
class ExogenousCorrectionHead(nn.Module):
    """
    Uses known future covariates, static context and the base PatchTST forecast
    to predict an additive correction for each future timestep.

    Inputs:
        future_known_reals: [B, prediction_length, F]
        static_context:     [B, hidden_dim]
        base_forecast:      [B, prediction_length]

    Output:
        correction:         [B, prediction_length]
    """
    def __init__(
        self,
        num_known_reals: int,
        static_context_dim: int,
        hidden_dim: int,
        dropout: float = 0.1,
        use_base_forecast: bool = True,
    ):
        super().__init__()

        if num_known_reals <= 0:
            raise ValueError("PatchTST-X requires known future real covariates.")

        self.use_base_forecast = use_base_forecast

        self.future_projection = nn.Sequential(
            nn.Linear(num_known_reals, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(hidden_dim),
        )

        correction_input_dim = hidden_dim + static_context_dim

        if use_base_forecast:
            correction_input_dim += 1

        self.correction_network = nn.Sequential(
            nn.Linear(correction_input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, future_known_reals, static_context, base_forecast=None):
        batch_size, prediction_length, _ = future_known_reals.shape

        future_context = self.future_projection(future_known_reals)

        static_context_time = static_context.unsqueeze(1).expand(
            -1,
            prediction_length,
            -1,
        )

        correction_parts = [
            future_context,
            static_context_time,
        ]

        if self.use_base_forecast:
            if base_forecast is None:
                raise ValueError("base_forecast must be provided when use_base_forecast=True.")

            correction_parts.append(base_forecast.unsqueeze(-1))

        correction_input = torch.cat(
            correction_parts,
            dim=-1,
        )

        correction = self.correction_network(correction_input).squeeze(-1)

        return correction

In [45]:
class PatchTSTXPointForecaster(nn.Module):
    """
    PatchTST-X point forecaster.

    Target path:
        past_target -> PatchTST backbone -> base forecast

    Exogenous path:
        static features + future known covariates + base forecast -> correction

    Final:
        forecast = base_forecast + correction_scale * correction
    """
    def __init__(
        self,
        context_length: int,
        prediction_length: int,
        static_cat_cardinalities: list[int],
        num_known_reals: int,
        num_static_reals: int,
        patch_length: int = 4,
        stride: int = 4,
        d_model: int = 64,
        num_layers: int = 2,
        attention_heads: int = 4,
        dim_feedforward: int | None = None,
        embedding_dim: int = 8,
        exog_hidden_dim: int = 64,
        dropout: float = 0.1,
        use_base_forecast_in_correction: bool = True,
        correction_scale_init: float = 0.1,
    ):
        super().__init__()

        self.context_length = context_length
        self.prediction_length = prediction_length
        self.num_known_reals = num_known_reals
        self.num_static_reals = num_static_reals
        self.use_base_forecast_in_correction = use_base_forecast_in_correction

        self.target_backbone = PatchTSTPointForecaster(
            context_length=context_length,
            prediction_length=prediction_length,
            patch_length=patch_length,
            stride=stride,
            d_model=d_model,
            num_layers=num_layers,
            attention_heads=attention_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
        )

        self.static_encoder = StaticFeatureEncoder(
            static_cat_cardinalities=static_cat_cardinalities,
            num_static_reals=num_static_reals,
            embedding_dim=embedding_dim,
            hidden_dim=exog_hidden_dim,
            dropout=dropout,
        )

        self.exogenous_correction = ExogenousCorrectionHead(
            num_known_reals=num_known_reals,
            static_context_dim=exog_hidden_dim,
            hidden_dim=exog_hidden_dim,
            dropout=dropout,
            use_base_forecast=use_base_forecast_in_correction,
        )

        self.correction_scale = nn.Parameter(
            torch.tensor(float(correction_scale_init))
        )

    def forward(
        self,
        past_target,
        past_known_reals=None,
        future_known_reals=None,
        static_categoricals=None,
        static_reals=None,
    ):
        base_forecast = self.target_backbone(
            past_target=past_target,
        )

        static_context = self.static_encoder(
            static_categoricals=static_categoricals,
            static_reals=static_reals,
        )

        correction = self.exogenous_correction(
            future_known_reals=future_known_reals,
            static_context=static_context,
            base_forecast=base_forecast,
        )

        forecast = base_forecast + self.correction_scale * correction

        return forecast

In [43]:
x_test_batch = next(iter(patchtst_last39_x_data["train_loader"]))
x_test_batch = move_batch_to_device(x_test_batch, DEVICE)

patchtst_x_test_model = PatchTSTXPointForecaster(
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    static_cat_cardinalities=patchtst_last39_x_data["static_cat_cardinalities"],
    num_known_reals=patchtst_last39_x_data["num_known_reals"],
    num_static_reals=patchtst_last39_x_data["num_static_reals"],
    patch_length=4,
    stride=2,
    d_model=64,
    num_layers=2,
    attention_heads=4,
    dim_feedforward=None,
    embedding_dim=8,
    exog_hidden_dim=64,
    dropout=0.1,
).to(DEVICE)

with torch.no_grad():
    x_test_output = patchtst_x_test_model(
        past_target=x_test_batch["past_target"],
        past_known_reals=x_test_batch["past_known_reals"],
        future_known_reals=x_test_batch["future_known_reals"],
        static_categoricals=x_test_batch["static_categoricals"],
        static_reals=x_test_batch["static_reals"],
    )

print("PatchTST-X output shape:", tuple(x_test_output.shape))
print("Expected shape:", tuple(x_test_batch["future_target"].shape))

assert x_test_output.shape == x_test_batch["future_target"].shape

print("Number of patches:", patchtst_x_test_model.target_backbone.num_patches)
print("Trainable parameters:", count_parameters(patchtst_x_test_model))

del patchtst_x_test_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("PatchTST-X shape test passed.")

PatchTST-X output shape: (256, 39)
Expected shape: (256, 39)
Number of patches: 25
Trainable parameters: 184840
PatchTST-X shape test passed.


## Training utilities

In [26]:
def forward_model(model, batch):
    return model(
        past_target=batch["past_target"],
        past_known_reals=batch["past_known_reals"],
        future_known_reals=batch["future_known_reals"],
        static_categoricals=batch["static_categoricals"],
        static_reals=batch["static_reals"],
    )


def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()

    total_loss = 0.0
    total_items = 0

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        optimizer.zero_grad(set_to_none=True)

        preds_scaled = forward_model(model, batch)
        target_scaled = batch["future_target"]

        try:
            loss = loss_fn(preds_scaled, target_scaled, batch)
        except TypeError:
            loss = loss_fn(preds_scaled, target_scaled)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        n_items = target_scaled.numel()
        total_loss += loss.item() * n_items
        total_items += n_items

    return total_loss / total_items

In [27]:
@torch.no_grad()
def evaluate_model(model, loader, loss_fn, device, holiday_feature_idx: int):
    model.eval()

    total_loss = 0.0
    total_items = 0

    all_true = []
    all_pred = []
    all_holiday = []

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        preds_scaled = forward_model(model, batch)
        target_scaled = batch["future_target"]

        try:
            loss = loss_fn(preds_scaled, target_scaled, batch)
        except TypeError:
            loss = loss_fn(preds_scaled, target_scaled)

        n_items = target_scaled.numel()
        total_loss += loss.item() * n_items
        total_items += n_items

        preds_original = inverse_scale_torch(
            preds_scaled,
            batch["target_mean"],
            batch["target_std"],
        )

        target_original = inverse_scale_torch(
            target_scaled,
            batch["target_mean"],
            batch["target_std"],
        )

        is_holiday = batch["future_known_reals"][:, :, holiday_feature_idx]

        all_pred.append(preds_original.detach().cpu().numpy())
        all_true.append(target_original.detach().cpu().numpy())
        all_holiday.append(is_holiday.detach().cpu().numpy())

    y_pred = np.concatenate(all_pred, axis=0)
    y_true = np.concatenate(all_true, axis=0)
    is_holiday = np.concatenate(all_holiday, axis=0)

    valid_loss = total_loss / total_items
    valid_wmae = weighted_mae_np(y_true, y_pred, is_holiday)
    valid_mae = np.mean(np.abs(y_true.reshape(-1) - y_pred.reshape(-1)))

    return {
        "valid_loss": valid_loss,
        "valid_wmae": valid_wmae,
        "valid_mae": valid_mae,
    }

In [28]:
class WeightedHuberLoss(nn.Module):
    def __init__(
        self,
        holiday_feature_idx: int,
        holiday_weight: float = 5.0,
        delta: float = 1.0,
    ):
        super().__init__()
        self.holiday_feature_idx = holiday_feature_idx
        self.holiday_weight = holiday_weight
        self.delta = delta

    def forward(self, preds_scaled, target_scaled, batch):
        element_loss = F.huber_loss(
            preds_scaled,
            target_scaled,
            reduction="none",
            delta=self.delta,
        )

        is_holiday = batch["future_known_reals"][:, :, self.holiday_feature_idx].bool()

        weights = torch.where(
            is_holiday,
            torch.full_like(element_loss, self.holiday_weight),
            torch.ones_like(element_loss),
        )

        return (weights * element_loss).sum() / weights.sum()

In [29]:
def fit_model(
    model,
    train_loader,
    valid_loader,
    optimizer,
    loss_fn,
    device,
    holiday_feature_idx: int,
    epochs: int,
    metric_prefix: str = "",
):
    best_state_dict = None
    best_valid_wmae = float("inf")
    best_epoch = None

    history = []

    prefix = f"{metric_prefix}_" if metric_prefix else ""

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=device,
        )

        valid_metrics = evaluate_model(
            model=model,
            loader=valid_loader,
            loss_fn=loss_fn,
            device=device,
            holiday_feature_idx=holiday_feature_idx,
        )

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            **valid_metrics,
        }
        history.append(row)

        if mlflow.active_run() is not None:
            mlflow.log_metric(f"{prefix}train_loss", train_loss, step=epoch)
            mlflow.log_metric(f"{prefix}valid_loss", valid_metrics["valid_loss"], step=epoch)
            mlflow.log_metric(f"{prefix}valid_wmae", valid_metrics["valid_wmae"], step=epoch)
            mlflow.log_metric(f"{prefix}valid_mae", valid_metrics["valid_mae"], step=epoch)

        if valid_metrics["valid_wmae"] < best_valid_wmae:
            best_valid_wmae = valid_metrics["valid_wmae"]
            best_epoch = epoch
            best_state_dict = deepcopy(model.state_dict())

        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_loss:.5f} | "
            f"valid_loss={valid_metrics['valid_loss']:.5f} | "
            f"valid_wmae={valid_metrics['valid_wmae']:.2f} | "
            f"valid_mae={valid_metrics['valid_mae']:.2f}"
        )

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    history_df = pd.DataFrame(history)

    return {
        "model": model,
        "history": history_df,
        "best_valid_wmae": best_valid_wmae,
        "best_epoch": best_epoch,
        "best_state_dict": best_state_dict,
    }

In [30]:
@torch.no_grad()
def collect_validation_predictions(
    model,
    loader,
    dataset,
    device,
    holiday_feature_idx: int,
):
    model.eval()

    all_true = []
    all_pred = []
    all_holiday = []

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        preds_scaled = forward_model(model, batch)
        target_scaled = batch["future_target"]

        preds_original = inverse_scale_torch(
            preds_scaled,
            batch["target_mean"],
            batch["target_std"],
        )

        target_original = inverse_scale_torch(
            target_scaled,
            batch["target_mean"],
            batch["target_std"],
        )

        is_holiday = batch["future_known_reals"][:, :, holiday_feature_idx]

        all_pred.append(preds_original.detach().cpu().numpy())
        all_true.append(target_original.detach().cpu().numpy())
        all_holiday.append(is_holiday.detach().cpu().numpy())

    y_pred = np.concatenate(all_pred, axis=0).reshape(-1)
    y_true = np.concatenate(all_true, axis=0).reshape(-1)
    is_holiday = np.concatenate(all_holiday, axis=0).reshape(-1).astype(bool)

    index_df = dataset.get_future_index().reset_index(drop=True).copy()

    assert len(index_df) == len(y_pred)

    index_df["y_true"] = y_true
    index_df["y_pred"] = y_pred
    index_df["abs_error"] = np.abs(index_df["y_true"] - index_df["y_pred"])
    index_df["IsHoliday_eval"] = is_holiday

    return index_df

In [31]:
def plot_loss_history(history_df, title, output_path):
    fig = plt.figure()
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    plt.plot(history_df["epoch"], history_df["valid_loss"], label="valid_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Scaled loss")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


def plot_wmae_history(history_df, title, output_path):
    fig = plt.figure()
    plt.plot(history_df["epoch"], history_df["valid_wmae"], label="valid_wmae")
    plt.xlabel("Epoch")
    plt.ylabel("Validation WMAE")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)

In [32]:
ARTIFACT_DIR = repo_root / "artifacts" / "patchtst"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

experiment_results = []

print("Artifact dir:", ARTIFACT_DIR)

Artifact dir: /kaggle/working/Walmart/artifacts/patchtst


## PatchTSTExperiments: Last-39 Validation 

In [34]:
def safe_float_name(x):
    return str(x).replace(".", "p").replace("-", "m")

In [ ]:
print("PatchTST target-only last-39 train windows:", len(patchtst_last39_target_data["train_dataset"]))
print("PatchTST target-only last-39 valid series:", len(patchtst_last39_target_data["valid_dataset"]))
print("PatchTST target-only last-39 train batches:", len(patchtst_last39_target_data["train_loader"]))
print("PatchTST target-only last-39 valid batches:", len(patchtst_last39_target_data["valid_loader"]))

print("\nPatchTST target-only calendar train windows:", len(patchtst_calendar_target_data["train_dataset"]))
print("PatchTST target-only calendar valid series:", len(patchtst_calendar_target_data["valid_dataset"]))
print("PatchTST target-only calendar train batches:", len(patchtst_calendar_target_data["train_loader"]))
print("PatchTST target-only calendar valid batches:", len(patchtst_calendar_target_data["valid_loader"]))

In [ ]:
PATCHTST_BASELINE_CONFIG = {
    "model": "PatchTSTPointForecaster",
    "model_variant": "target_only",
    "feature_set": "past_target_only_with_IsHoliday_for_metric",
    "validation_strategy": "last_39_weeks",
    "context_length": CONTEXT_LENGTH,
    "prediction_length": PREDICTION_LENGTH,
    "patch_length": 4,
    "stride": 4,
    "d_model": 64,
    "num_layers": 2,
    "attention_heads": 4,
    "dim_feedforward": None,
    "dropout": 0.1,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "loss_name": "huber",
    "batch_size": BATCH_SIZE,
    "epochs": 10,
    "optimizer": "AdamW",
}

In [ ]:
set_seed(SEED)

config = PATCHTST_BASELINE_CONFIG.copy()

run_name = (
    f"PatchTST_Last39_Baseline_"
    f"p{config['patch_length']}_"
    f"s{config['stride']}_"
    f"d{config['d_model']}_"
    f"layers{config['num_layers']}_"
    f"heads{config['attention_heads']}_"
    f"drop{safe_float_name(config['dropout'])}_"
    f"lr{safe_float_name(config['lr'])}_"
    f"wd{safe_float_name(config['weight_decay'])}_"
    f"{config['loss_name']}"
)

print(run_name)
print(config)

model = PatchTSTPointForecaster(
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    patch_length=config["patch_length"],
    stride=config["stride"],
    d_model=config["d_model"],
    num_layers=config["num_layers"],
    attention_heads=config["attention_heads"],
    dim_feedforward=config["dim_feedforward"],
    dropout=config["dropout"],
).to(DEVICE)

print(model)
print("Trainable parameters:", count_parameters(model))
print("Number of patches:", model.num_patches)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["lr"],
    weight_decay=config["weight_decay"],
)

loss_fn = make_loss_fn(config["loss_name"])

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name=run_name) as run:
    run_id = run.info.run_id

    mlflow.log_params({
        **config,
        "train_loader_type": "FastTensorDataLoader",
        "train_windows": len(patchtst_last39_target_data["train_dataset"]),
        "valid_series": len(patchtst_last39_target_data["valid_dataset"]),
        "num_known_reals": patchtst_last39_target_data["num_known_reals"],
        "num_static_reals": patchtst_last39_target_data["num_static_reals"],
        "static_cat_cardinalities": str(patchtst_last39_target_data["static_cat_cardinalities"]),
        "num_patches": model.num_patches,
        "trainable_parameters": count_parameters(model),
    })

    result = fit_model(
        model=model,
        train_loader=patchtst_last39_target_data["train_loader"],
        valid_loader=patchtst_last39_target_data["valid_loader"],
        optimizer=optimizer,
        loss_fn=loss_fn,
        device=DEVICE,
        holiday_feature_idx=patchtst_last39_target_data["holiday_feature_idx"],
        epochs=config["epochs"],
        metric_prefix="last39",
    )

    mlflow.log_metric("last39_best_valid_wmae", result["best_valid_wmae"])
    mlflow.log_metric("last39_best_epoch", result["best_epoch"])

print("Run ID:", run_id)
print("Best last-39 WMAE:", result["best_valid_wmae"])
print("Best epoch:", result["best_epoch"])

patchtst_baseline_result = {
    "run_name": run_name,
    "run_id": run_id,
    **config,
    "best_valid_wmae": result["best_valid_wmae"],
    "best_epoch": result["best_epoch"],
    "trainable_parameters": count_parameters(model),
    "num_patches": model.num_patches,
}

experiment_results.append(patchtst_baseline_result)

display(result["history"])

In [ ]:
PATCHTST_LAST39_GRID = [
    # baseline, longer run
    {
        "patch_length": 4,
        "stride": 4,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # slightly more regularization
    {
        "patch_length": 4,
        "stride": 4,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # overlapping short patches
    {
        "patch_length": 4,
        "stride": 2,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # longer patches, overlapping
    {
        "patch_length": 8,
        "stride": 4,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # longer patches, non-overlapping
    {
        "patch_length": 8,
        "stride": 8,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # lower learning rate version of baseline
    {
        "patch_length": 4,
        "stride": 4,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "dropout": 0.1,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # wider model
    {
        "patch_length": 4,
        "stride": 4,
        "d_model": 128,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },

    # deeper model
    {
        "patch_length": 4,
        "stride": 4,
        "d_model": 64,
        "num_layers": 3,
        "attention_heads": 4,
        "dim_feedforward": None,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
    },
]

PATCHTST_LAST39_EPOCHS = 25

In [ ]:
set_seed(SEED)

patchtst_last39_results = []

START_FROM_CONFIG = 4

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

for i, config in enumerate(PATCHTST_LAST39_GRID, start=1):
    if i < START_FROM_CONFIG:
        continue

    run_name = (
        f"PatchTST_Last39_"
        f"p{config['patch_length']}_"
        f"s{config['stride']}_"
        f"d{config['d_model']}_"
        f"layers{config['num_layers']}_"
        f"heads{config['attention_heads']}_"
        f"drop{safe_float_name(config['dropout'])}_"
        f"lr{safe_float_name(config['lr'])}_"
        f"wd{safe_float_name(config['weight_decay'])}_"
        f"{config['loss_name']}"
    )

    print("=" * 100)
    print(f"Run {i}/{len(PATCHTST_LAST39_GRID)}: {run_name}")
    print(config)

    set_seed(SEED + i)

    model = PatchTSTPointForecaster(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        patch_length=config["patch_length"],
        stride=config["stride"],
        d_model=config["d_model"],
        num_layers=config["num_layers"],
        attention_heads=config["attention_heads"],
        dim_feedforward=config["dim_feedforward"],
        dropout=config["dropout"],
    ).to(DEVICE)

    n_params = count_parameters(model)
    print("Trainable parameters:", n_params)
    print("Number of patches:", model.num_patches)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "PatchTSTPointForecaster",
            "model_variant": "target_only",
            "feature_set": "past_target_only_with_IsHoliday_for_metric",
            "validation_strategy": "last_39_weeks",
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "patch_length": config["patch_length"],
            "stride": config["stride"],
            "num_patches": model.num_patches,
            "d_model": config["d_model"],
            "num_layers": config["num_layers"],
            "attention_heads": config["attention_heads"],
            "dim_feedforward": config["dim_feedforward"],
            "dropout": config["dropout"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "batch_size": BATCH_SIZE,
            "epochs": PATCHTST_LAST39_EPOCHS,
            "optimizer": "AdamW",
            "train_loader_type": "FastTensorDataLoader",
            "train_windows": len(patchtst_last39_target_data["train_dataset"]),
            "valid_series": len(patchtst_last39_target_data["valid_dataset"]),
            "num_known_reals": patchtst_last39_target_data["num_known_reals"],
            "num_static_reals": patchtst_last39_target_data["num_static_reals"],
            "static_cat_cardinalities": str(patchtst_last39_target_data["static_cat_cardinalities"]),
            "trainable_parameters": n_params,
        })

        result = fit_model(
            model=model,
            train_loader=patchtst_last39_target_data["train_loader"],
            valid_loader=patchtst_last39_target_data["valid_loader"],
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            holiday_feature_idx=patchtst_last39_target_data["holiday_feature_idx"],
            epochs=PATCHTST_LAST39_EPOCHS,
            metric_prefix="last39",
        )

        best_wmae = result["best_valid_wmae"]
        best_epoch = result["best_epoch"]

        mlflow.log_metric("last39_best_valid_wmae", best_wmae)
        mlflow.log_metric("last39_best_epoch", best_epoch)

    patchtst_last39_results.append({
        "run_name": run_name,
        "run_id": run_id,
        **config,
        "num_patches": model.num_patches,
        "trainable_parameters": n_params,
        "last39_best_valid_wmae": best_wmae,
        "last39_best_epoch": best_epoch,
    })

    print(f"Best last-39 WMAE: {best_wmae:.2f}")
    print(f"Best epoch: {best_epoch}")

    del model
    del optimizer
    del loss_fn

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

patchtst_last39_results_df = (
    pd.DataFrame(patchtst_last39_results)
    .sort_values("last39_best_valid_wmae")
    .reset_index(drop=True)
)

display(patchtst_last39_results_df)

last39_results_path = ARTIFACT_DIR / "patchtst_last39_grid_results.csv"
patchtst_last39_results_df.to_csv(last39_results_path, index=False)

print("Saved:", last39_results_path)

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

runs = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    output_format="pandas",
)

patchtst_last39_runs = runs[
    (runs["params.model"] == "PatchTSTPointForecaster")
    & (runs["params.model_variant"] == "target_only")
    & (runs["params.validation_strategy"] == "last_39_weeks")
    & (runs["tags.mlflow.runName"].astype(str).str.startswith("PatchTST_Last39_"))
    & (~runs["tags.mlflow.runName"].astype(str).str.contains("Baseline", na=False))
].copy()

patchtst_last39_runs = patchtst_last39_runs[
    patchtst_last39_runs["metrics.last39_best_valid_wmae"].notna()
].copy()

print("Loaded PatchTST last-39 grid runs from MLflow:", len(patchtst_last39_runs))

display(
    patchtst_last39_runs[
        [
            "tags.mlflow.runName",
            "run_id",
            "metrics.last39_best_valid_wmae",
            "metrics.last39_best_epoch",
            "params.patch_length",
            "params.stride",
            "params.d_model",
            "params.num_layers",
            "params.attention_heads",
            "params.dropout",
            "params.learning_rate",
            "params.weight_decay",
            "params.loss_name",
            "params.trainable_parameters",
            "params.num_patches",
        ]
    ].sort_values("metrics.last39_best_valid_wmae")
)

In [ ]:
def parse_optional_int(value):
    if pd.isna(value) or value in [None, "None", "nan", ""]:
        return None
    return int(value)


patchtst_last39_results_df = pd.DataFrame({
    "run_name": patchtst_last39_runs["tags.mlflow.runName"],
    "run_id": patchtst_last39_runs["run_id"],
    "patch_length": patchtst_last39_runs["params.patch_length"].astype(int),
    "stride": patchtst_last39_runs["params.stride"].astype(int),
    "d_model": patchtst_last39_runs["params.d_model"].astype(int),
    "num_layers": patchtst_last39_runs["params.num_layers"].astype(int),
    "attention_heads": patchtst_last39_runs["params.attention_heads"].astype(int),
    "dim_feedforward": patchtst_last39_runs["params.dim_feedforward"].apply(parse_optional_int),
    "dropout": patchtst_last39_runs["params.dropout"].astype(float),
    "lr": patchtst_last39_runs["params.learning_rate"].astype(float),
    "weight_decay": patchtst_last39_runs["params.weight_decay"].astype(float),
    "loss_name": patchtst_last39_runs["params.loss_name"],
    "num_patches": patchtst_last39_runs["params.num_patches"].astype(int),
    "trainable_parameters": patchtst_last39_runs["params.trainable_parameters"].astype(int),
    "last39_best_valid_wmae": patchtst_last39_runs["metrics.last39_best_valid_wmae"].astype(float),
    "last39_best_epoch": patchtst_last39_runs["metrics.last39_best_epoch"].astype(int),
})

config_subset = [
    "patch_length",
    "stride",
    "d_model",
    "num_layers",
    "attention_heads",
    "dim_feedforward",
    "dropout",
    "lr",
    "weight_decay",
    "loss_name",
]

# keep the one with the best WMAE
patchtst_last39_results_df = (
    patchtst_last39_results_df
    .sort_values("last39_best_valid_wmae")
    .drop_duplicates(subset=config_subset, keep="first")
    .reset_index(drop=True)
)

display(patchtst_last39_results_df)

last39_results_path = ARTIFACT_DIR / "patchtst_last39_grid_results_from_mlflow.csv"
patchtst_last39_results_df.to_csv(last39_results_path, index=False)

print("Saved:", last39_results_path)

In [ ]:
top_configs = (
    patchtst_last39_results_df
    .sort_values("last39_best_valid_wmae")
    .head(3)
    .copy()
)

diverse_configs = []

# best short patch non-overlapping setup
diverse_configs.append(
    patchtst_last39_results_df[
        (patchtst_last39_results_df["patch_length"] == 4)
        & (patchtst_last39_results_df["stride"] == 4)
    ].sort_values("last39_best_valid_wmae").head(1)
)

# best overlapping setup
diverse_configs.append(
    patchtst_last39_results_df[
        patchtst_last39_results_df["stride"] < patchtst_last39_results_df["patch_length"]
    ].sort_values("last39_best_valid_wmae").head(1)
)

# best longer patch setup
diverse_configs.append(
    patchtst_last39_results_df[
        patchtst_last39_results_df["patch_length"] == 8
    ].sort_values("last39_best_valid_wmae").head(1)
)

# best wider model
diverse_configs.append(
    patchtst_last39_results_df[
        patchtst_last39_results_df["d_model"] == 128
    ].sort_values("last39_best_valid_wmae").head(1)
)

patchtst_calendar_candidate_configs = pd.concat(
    [top_configs] + diverse_configs,
    axis=0,
).drop_duplicates(
    subset=[
        "patch_length",
        "stride",
        "d_model",
        "num_layers",
        "attention_heads",
        "dim_feedforward",
        "dropout",
        "lr",
        "weight_decay",
        "loss_name",
    ]
).sort_values(
    "last39_best_valid_wmae"
).reset_index(drop=True)

display(patchtst_calendar_candidate_configs)
print("Calendar candidate configs:", len(patchtst_calendar_candidate_configs))

### PatchTST-X

In [46]:
print("PatchTST-X last-39 train windows:", len(patchtst_last39_x_data["train_dataset"]))
print("PatchTST-X last-39 valid series:", len(patchtst_last39_x_data["valid_dataset"]))
print("PatchTST-X last-39 train batches:", len(patchtst_last39_x_data["train_loader"]))
print("PatchTST-X last-39 valid batches:", len(patchtst_last39_x_data["valid_loader"]))

print("\nPatchTST-X feature dimensions:")
print("static cat cardinalities:", patchtst_last39_x_data["static_cat_cardinalities"])
print("num static reals:", patchtst_last39_x_data["num_static_reals"])
print("num known future reals:", patchtst_last39_x_data["num_known_reals"])
print("holiday idx:", patchtst_last39_x_data["holiday_feature_idx"])

print("\nKnown future real columns:")
print(patchtst_last39_x_data["cols"]["known_future_real_cols"])

print("\nStatic categorical columns:")
print(patchtst_last39_x_data["cols"]["static_cat_cols"])

print("\nStatic real columns:")
print(patchtst_last39_x_data["cols"]["static_real_cols"])

PatchTST-X last-39 train windows: 38464
PatchTST-X last-39 valid series: 2762
PatchTST-X last-39 train batches: 151
PatchTST-X last-39 valid batches: 11

PatchTST-X feature dimensions:
static cat cardinalities: [46, 82, 4]
num static reals: 1
num known future reals: 11
holiday idx: 6

Known future real columns:
['Temperature_scaled', 'Fuel_Price_scaled', 'CPI_scaled', 'Unemployment_scaled', 'Week_sin_scaled', 'Week_cos_scaled', 'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas']

Static categorical columns:
['Store_id', 'Dept_id', 'Type_id']

Static real columns:
['Size_scaled']


In [47]:
PATCHTST_X_LAST39_GRID = [
    # best target-only last39 family now with X correction
    {
        "patch_length": 8,
        "stride": 8,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "embedding_dim": 8,
        "exog_hidden_dim": 64,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
        "use_base_forecast_in_correction": True,
        "correction_scale_init": 0.1,
    },

    # best calendar target-only family
    {
        "patch_length": 4,
        "stride": 2,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "embedding_dim": 8,
        "exog_hidden_dim": 64,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
        "use_base_forecast_in_correction": True,
        "correction_scale_init": 0.1,
    },

    # short non-overlapping baseline family
    {
        "patch_length": 4,
        "stride": 4,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "embedding_dim": 8,
        "exog_hidden_dim": 64,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
        "use_base_forecast_in_correction": True,
        "correction_scale_init": 0.1,
    },

    # longer overlapping patch setup
    {
        "patch_length": 8,
        "stride": 4,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "embedding_dim": 8,
        "exog_hidden_dim": 64,
        "dropout": 0.1,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "loss_name": "huber",
        "use_base_forecast_in_correction": True,
        "correction_scale_init": 0.1,
    },

    # regularized version of the best calendar family
    {
        "patch_length": 4,
        "stride": 2,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "embedding_dim": 8,
        "exog_hidden_dim": 64,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "loss_name": "huber",
        "use_base_forecast_in_correction": True,
        "correction_scale_init": 0.1,
    },

    # regularized version of the best last39 target-only family
    {
        "patch_length": 8,
        "stride": 8,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "embedding_dim": 8,
        "exog_hidden_dim": 64,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "loss_name": "huber",
        "use_base_forecast_in_correction": True,
        "correction_scale_init": 0.1,
    },

    # smaller Transformer backbone, same X correction
    {
        "patch_length": 4,
        "stride": 2,
        "d_model": 64,
        "num_layers": 1,
        "attention_heads": 4,
        "dim_feedforward": 128,
        "embedding_dim": 8,
        "exog_hidden_dim": 64,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "loss_name": "huber",
        "use_base_forecast_in_correction": True,
        "correction_scale_init": 0.1,
    },

    # smaller X correction head, to check overfitting
    {
        "patch_length": 4,
        "stride": 2,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": 128,
        "embedding_dim": 8,
        "exog_hidden_dim": 32,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "loss_name": "huber",
        "use_base_forecast_in_correction": True,
        "correction_scale_init": 0.1,
    },
]

PATCHTST_X_LAST39_EPOCHS = 25

print("PatchTST-X last-39 configs:", len(PATCHTST_X_LAST39_GRID))
print("Epochs per config:", PATCHTST_X_LAST39_EPOCHS)

PatchTST-X last-39 configs: 8
Epochs per config: 25


In [48]:
set_seed(SEED)

patchtst_x_last39_results = []

START_FROM_CONFIG = 1

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

for i, config in enumerate(PATCHTST_X_LAST39_GRID, start=1):
    if i < START_FROM_CONFIG:
        continue

    run_name = (
        f"PatchTSTX_Last39_"
        f"p{config['patch_length']}_"
        f"s{config['stride']}_"
        f"d{config['d_model']}_"
        f"layers{config['num_layers']}_"
        f"heads{config['attention_heads']}_"
        f"exog{config['exog_hidden_dim']}_"
        f"drop{safe_float_name(config['dropout'])}_"
        f"lr{safe_float_name(config['lr'])}_"
        f"wd{safe_float_name(config['weight_decay'])}_"
        f"{config['loss_name']}"
    )

    print("=" * 100)
    print(f"Run {i}/{len(PATCHTST_X_LAST39_GRID)}: {run_name}")
    print(config)

    set_seed(SEED + 200 + i)

    model = PatchTSTXPointForecaster(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        static_cat_cardinalities=patchtst_last39_x_data["static_cat_cardinalities"],
        num_known_reals=patchtst_last39_x_data["num_known_reals"],
        num_static_reals=patchtst_last39_x_data["num_static_reals"],
        patch_length=config["patch_length"],
        stride=config["stride"],
        d_model=config["d_model"],
        num_layers=config["num_layers"],
        attention_heads=config["attention_heads"],
        dim_feedforward=config["dim_feedforward"],
        embedding_dim=config["embedding_dim"],
        exog_hidden_dim=config["exog_hidden_dim"],
        dropout=config["dropout"],
        use_base_forecast_in_correction=config["use_base_forecast_in_correction"],
        correction_scale_init=config["correction_scale_init"],
    ).to(DEVICE)

    n_params = count_parameters(model)

    print("Trainable parameters:", n_params)
    print("Number of patches:", model.target_backbone.num_patches)
    print("Initial correction scale:", float(model.correction_scale.detach().cpu()))

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "PatchTSTXPointForecaster",
            "model_variant": "patchtst_x_correction",
            "feature_set": "stable_no_markdowns_exogenous_correction",
            "validation_strategy": "last_39_weeks",
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "patch_length": config["patch_length"],
            "stride": config["stride"],
            "num_patches": model.target_backbone.num_patches,
            "d_model": config["d_model"],
            "num_layers": config["num_layers"],
            "attention_heads": config["attention_heads"],
            "dim_feedforward": config["dim_feedforward"],
            "embedding_dim": config["embedding_dim"],
            "exog_hidden_dim": config["exog_hidden_dim"],
            "dropout": config["dropout"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "batch_size": BATCH_SIZE,
            "epochs": PATCHTST_X_LAST39_EPOCHS,
            "optimizer": "AdamW",
            "train_loader_type": "FastTensorDataLoader",
            "train_windows": len(patchtst_last39_x_data["train_dataset"]),
            "valid_series": len(patchtst_last39_x_data["valid_dataset"]),
            "num_known_reals": patchtst_last39_x_data["num_known_reals"],
            "num_static_reals": patchtst_last39_x_data["num_static_reals"],
            "static_cat_cardinalities": str(patchtst_last39_x_data["static_cat_cardinalities"]),
            "use_base_forecast_in_correction": config["use_base_forecast_in_correction"],
            "correction_scale_init": config["correction_scale_init"],
            "trainable_parameters": n_params,
        })

        result = fit_model(
            model=model,
            train_loader=patchtst_last39_x_data["train_loader"],
            valid_loader=patchtst_last39_x_data["valid_loader"],
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            holiday_feature_idx=patchtst_last39_x_data["holiday_feature_idx"],
            epochs=PATCHTST_X_LAST39_EPOCHS,
            metric_prefix="last39",
        )

        best_wmae = result["best_valid_wmae"]
        best_epoch = result["best_epoch"]

        mlflow.log_metric("last39_best_valid_wmae", best_wmae)
        mlflow.log_metric("last39_best_epoch", best_epoch)
        mlflow.log_metric("final_correction_scale", float(model.correction_scale.detach().cpu()))

    patchtst_x_last39_results.append({
        "run_name": run_name,
        "run_id": run_id,
        **config,
        "num_patches": model.target_backbone.num_patches,
        "trainable_parameters": n_params,
        "last39_best_valid_wmae": best_wmae,
        "last39_best_epoch": best_epoch,
        "final_correction_scale": float(model.correction_scale.detach().cpu()),
    })

    print(f"Best last-39 WMAE: {best_wmae:.2f}")
    print(f"Best epoch: {best_epoch}")
    print("Final correction scale:", float(model.correction_scale.detach().cpu()))

    del model
    del optimizer
    del loss_fn

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

patchtst_x_last39_results_df = (
    pd.DataFrame(patchtst_x_last39_results)
    .sort_values("last39_best_valid_wmae")
    .reset_index(drop=True)
)

display(patchtst_x_last39_results_df)

x_last39_results_path = ARTIFACT_DIR / "patchtst_x_last39_grid_results.csv"
patchtst_x_last39_results_df.to_csv(x_last39_results_path, index=False)

print("Saved:", x_last39_results_path)

Run 1/8: PatchTSTX_Last39_p8_s8_d64_layers2_heads4_exog64_drop0p1_lr0p001_wd0p0001_huber
{'patch_length': 8, 'stride': 8, 'd_model': 64, 'num_layers': 2, 'attention_heads': 4, 'dim_feedforward': None, 'embedding_dim': 8, 'exog_hidden_dim': 64, 'dropout': 0.1, 'lr': 0.001, 'weight_decay': 0.0001, 'loss_name': 'huber', 'use_base_forecast_in_correction': True, 'correction_scale_init': 0.1}
Trainable parameters: 139081
Number of patches: 7
Initial correction scale: 0.10000000149011612
Epoch 001 | train_loss=0.21001 | valid_loss=0.40716 | valid_wmae=2426.50 | valid_mae=2409.76
Epoch 002 | train_loss=0.16429 | valid_loss=0.38824 | valid_wmae=2318.18 | valid_mae=2276.23
Epoch 003 | train_loss=0.15538 | valid_loss=0.39507 | valid_wmae=2345.20 | valid_mae=2307.41
Epoch 004 | train_loss=0.15052 | valid_loss=0.39038 | valid_wmae=2336.07 | valid_mae=2292.12
Epoch 005 | train_loss=0.14721 | valid_loss=0.38207 | valid_wmae=2279.23 | valid_mae=2244.80
Epoch 006 | train_loss=0.14450 | valid_loss=0.391

,run_name,run_id,patch_length,stride,d_model,num_layers,attention_heads,dim_feedforward,embedding_dim,exog_hidden_dim,...,lr,weight_decay,loss_name,use_base_forecast_in_correction,correction_scale_init,num_patches,trainable_parameters,last39_best_valid_wmae,last39_best_epoch,final_correction_scale
0,PatchTSTX_Last39_p8_s8_d64_layers2_heads4_exog...,a91c8c59c4d74ba89cf01f108770235d,8,8,64,2,4,NaN,8,64,...,0.0010,0.0001,huber,True,0.1,7,139081,2237.879817,19,0.360847
1,PatchTSTX_Last39_p4_s4_d64_layers2_heads4_exog...,71ab4d43d7474285b86c7a2c8077947b,4,4,64,2,4,NaN,8,64,...,0.0010,0.0001,huber,True,0.1,13,154185,2267.005818,12,0.339403
2,PatchTSTX_Last39_p8_s4_d64_layers2_heads4_exog...,e1151a8eb60341ef8abf6529e42c64d5,8,4,64,2,4,NaN,8,64,...,0.0010,0.0001,huber,True,0.1,12,151881,2277.168950,20,0.365842
3,PatchTSTX_Last39_p4_s2_d64_layers2_heads4_exog...,e94938b9a32f4dbdbaa850f50d952e50,4,2,64,2,4,NaN,8,64,...,0.0010,0.0001,huber,True,0.1,25,184905,2277.448482,22,0.344545
4,PatchTSTX_Last39_p8_s8_d64_layers2_heads4_exog...,d894a63ad207493bb7b28ae55b3d6d55,8,8,64,2,4,NaN,8,64,...,0.0005,0.0005,huber,True,0.1,7,139081,2283.560396,19,0.304305
5,PatchTSTX_Last39_p4_s2_d64_layers2_heads4_exog...,88bcee82e7cd4b0ca0451317c814a00a,4,2,64,2,4,NaN,8,64,...,0.0005,0.0005,huber,True,0.1,25,184905,2309.348868,11,0.270900
6,PatchTSTX_Last39_p4_s2_d64_layers2_heads4_exog...,8abea6d492a74c00a9ccf5a61d5cfbfd,4,2,64,2,4,128.0,8,32,...,0.0005,0.0005,huber,True,0.1,25,138089,2363.205487,25,0.328593
7,PatchTSTX_Last39_p4_s2_d64_layers1_heads4_exog...,9935ef8a5e3a4fb996a7c54e313d55d2,4,2,64,1,4,128.0,8,64,...,0.0005,0.0005,huber,True,0.1,25,118409,2368.938770,16,0.300828


Saved: /kaggle/working/Walmart/artifacts/patchtst/patchtst_x_last39_grid_results.csv


## PatchTST Experiments: Calendar-aligned 

In [37]:
PATCHTST_CALENDAR_EPOCHS = 50

patchtst_calendar_results = []

START_FROM_CONFIG = 1

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)


def parse_dim_feedforward(value):
    if pd.isna(value) or value in [None, "None", "nan", ""]:
        return None
    return int(value)


# print("Calendar candidate configs:", len(patchtst_calendar_candidate_configs))
# print("Epochs per config:", PATCHTST_CALENDAR_EPOCHS)

PATCHTST_CALENDAR_STABILITY_GRID = [
    # current best family, more regularized
    {
        "patch_length": 4,
        "stride": 2,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "loss_name": "huber",
    },

    # current best family, even stronger regularization
    {
        "patch_length": 4,
        "stride": 2,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": None,
        "dropout": 0.3,
        "lr": 5e-4,
        "weight_decay": 1e-3,
        "loss_name": "huber",
    },

    # same best family, smaller feedforward block
    {
        "patch_length": 4,
        "stride": 2,
        "d_model": 64,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": 128,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "loss_name": "huber",
    },

    # lower capacity version
    {
        "patch_length": 4,
        "stride": 2,
        "d_model": 32,
        "num_layers": 2,
        "attention_heads": 4,
        "dim_feedforward": 128,
        "dropout": 0.2,
        "lr": 1e-3,
        "weight_decay": 5e-4,
        "loss_name": "huber",
    },

    # simpler one layer version
    {
        "patch_length": 4,
        "stride": 2,
        "d_model": 64,
        "num_layers": 1,
        "attention_heads": 4,
        "dim_feedforward": 128,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "loss_name": "huber",
    },

    # best last-39 p8/s8 family, but regularized for calendar
    {
        "patch_length": 8,
        "stride": 8,
        "d_model": 64,
        "num_layers": 1,
        "attention_heads": 4,
        "dim_feedforward": 128,
        "dropout": 0.2,
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "loss_name": "huber",
    },
]

PATCHTST_CALENDAR_STABILITY_EPOCHS = 30
PATCHTST_CALENDAR_EPOCHS = PATCHTST_CALENDAR_STABILITY_EPOCHS

print("PatchTST calendar stability configs:", len(PATCHTST_CALENDAR_STABILITY_GRID))
print("Epochs per config:", PATCHTST_CALENDAR_STABILITY_EPOCHS)

# for i, row in patchtst_calendar_candidate_configs.iterrows():
for i, row in enumerate(PATCHTST_CALENDAR_STABILITY_GRID, start=1):
    run_number = i

    if run_number < START_FROM_CONFIG:
        continue

    config = {
        "patch_length": int(row["patch_length"]),
        "stride": int(row["stride"]),
        "d_model": int(row["d_model"]),
        "num_layers": int(row["num_layers"]),
        "attention_heads": int(row["attention_heads"]),
        "dim_feedforward": parse_dim_feedforward(row["dim_feedforward"]),
        "dropout": float(row["dropout"]),
        "lr": float(row["lr"]),
        "weight_decay": float(row["weight_decay"]),
        "loss_name": row["loss_name"],
    }

    run_name = (
        f"PatchTST_CalendarAligned_"
        f"p{config['patch_length']}_"
        f"s{config['stride']}_"
        f"d{config['d_model']}_"
        f"layers{config['num_layers']}_"
        f"heads{config['attention_heads']}_"
        f"drop{safe_float_name(config['dropout'])}_"
        f"lr{safe_float_name(config['lr'])}_"
        f"wd{safe_float_name(config['weight_decay'])}_"
        f"{config['loss_name']}"
    )

    print("=" * 100)
    print(f"Calendar run {run_number}/{len(PATCHTST_CALENDAR_STABILITY_GRID)}: {run_name}")
    print(config)

    set_seed(SEED + 100 + run_number)

    model = PatchTSTPointForecaster(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        patch_length=config["patch_length"],
        stride=config["stride"],
        d_model=config["d_model"],
        num_layers=config["num_layers"],
        attention_heads=config["attention_heads"],
        dim_feedforward=config["dim_feedforward"],
        dropout=config["dropout"],
    ).to(DEVICE)

    n_params = count_parameters(model)

    print("Trainable parameters:", n_params)
    print("Number of patches:", model.num_patches)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "PatchTSTPointForecaster",
            "model_variant": "target_only",
            "feature_set": "past_target_only_with_IsHoliday_for_metric",
            "validation_strategy": "calendar_aligned_39_weeks",
            "calendar_valid_start": "2011-11-04",
            "calendar_valid_end": "2012-07-27",
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "patch_length": config["patch_length"],
            "stride": config["stride"],
            "num_patches": model.num_patches,
            "d_model": config["d_model"],
            "num_layers": config["num_layers"],
            "attention_heads": config["attention_heads"],
            "dim_feedforward": config["dim_feedforward"],
            "dropout": config["dropout"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "batch_size": BATCH_SIZE,
            "epochs": PATCHTST_CALENDAR_EPOCHS,
            "optimizer": "AdamW",
            "train_loader_type": "FastTensorDataLoader",
            "train_windows": len(patchtst_calendar_target_data["train_dataset"]),
            "valid_series": len(patchtst_calendar_target_data["valid_dataset"]),
            "num_known_reals": patchtst_calendar_target_data["num_known_reals"],
            "num_static_reals": patchtst_calendar_target_data["num_static_reals"],
            "static_cat_cardinalities": str(patchtst_calendar_target_data["static_cat_cardinalities"]),
            "trainable_parameters": n_params,
            # "source_last39_run_id": row["run_id"],
            # "source_last39_best_valid_wmae": float(row["last39_best_valid_wmae"]),
            # "source_last39_best_epoch": int(row["last39_best_epoch"]),
        })

        result = fit_model(
            model=model,
            train_loader=patchtst_calendar_target_data["train_loader"],
            valid_loader=patchtst_calendar_target_data["valid_loader"],
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            holiday_feature_idx=patchtst_calendar_target_data["holiday_feature_idx"],
            epochs=PATCHTST_CALENDAR_EPOCHS,
            metric_prefix="calendar",
        )

        best_wmae = result["best_valid_wmae"]
        best_epoch = result["best_epoch"]

        mlflow.log_metric("calendar_best_valid_wmae", best_wmae)
        mlflow.log_metric("calendar_best_epoch", best_epoch)

    patchtst_calendar_results.append({
        "run_name": run_name,
        "run_id": run_id,
        **config,
        "num_patches": model.num_patches,
        "trainable_parameters": n_params,
        # "source_last39_run_id": row["run_id"],
        # "source_last39_best_valid_wmae": float(row["last39_best_valid_wmae"]),
        # "source_last39_best_epoch": int(row["last39_best_epoch"]),
        "calendar_best_valid_wmae": best_wmae,
        "calendar_best_epoch": best_epoch,
    })

    print(f"Best calendar WMAE: {best_wmae:.2f}")
    print(f"Best epoch: {best_epoch}")

    del model
    del optimizer
    del loss_fn

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

patchtst_calendar_results_df = (
    pd.DataFrame(patchtst_calendar_results)
    .sort_values("calendar_best_valid_wmae")
    .reset_index(drop=True)
)

display(patchtst_calendar_results_df)

calendar_results_path = ARTIFACT_DIR / "patchtst_calendar_check_results.csv"
patchtst_calendar_results_df.to_csv(calendar_results_path, index=False)

print("Saved:", calendar_results_path)

PatchTST calendar stability configs: 6
Epochs per config: 30
Calendar run 2/6: PatchTST_CalendarAligned_p4_s2_d64_layers2_heads4_drop0p2_lr0p0005_wd0p0005_huber
{'patch_length': 4, 'stride': 2, 'd_model': 64, 'num_layers': 2, 'attention_heads': 4, 'dim_feedforward': None, 'dropout': 0.2, 'lr': 0.0005, 'weight_decay': 0.0005, 'loss_name': 'huber'}
Trainable parameters: 164455
Number of patches: 25
Epoch 001 | train_loss=0.33294 | valid_loss=0.59993 | valid_wmae=3777.11 | valid_mae=3341.84
Epoch 002 | train_loss=0.24046 | valid_loss=0.56402 | valid_wmae=3540.37 | valid_mae=3109.11
Epoch 003 | train_loss=0.21929 | valid_loss=0.56974 | valid_wmae=3494.75 | valid_mae=3110.35
Epoch 004 | train_loss=0.20731 | valid_loss=0.56847 | valid_wmae=3471.79 | valid_mae=3088.03
Epoch 005 | train_loss=0.19918 | valid_loss=0.56930 | valid_wmae=3454.08 | valid_mae=3083.62
Epoch 006 | train_loss=0.19322 | valid_loss=0.56699 | valid_wmae=3442.35 | valid_mae=3065.67
Epoch 007 | train_loss=0.18771 | valid_los

,run_name,run_id,patch_length,stride,d_model,num_layers,attention_heads,dim_feedforward,dropout,lr,weight_decay,loss_name,num_patches,trainable_parameters,calendar_best_valid_wmae,calendar_best_epoch
0,PatchTST_CalendarAligned_p4_s2_d64_layers2_hea...,4e81cfb19c0347d6a326dfd825a31d28,4,2,64,2,4,NaN,0.2,0.0005,0.0005,huber,25,164455,3407.443391,7
1,PatchTST_CalendarAligned_p4_s2_d64_layers2_hea...,38cc0dbd02fd4e3e8559a92b19e3b7cd,4,2,64,2,4,NaN,0.3,0.0005,0.0010,huber,25,164455,3408.773414,9
2,PatchTST_CalendarAligned_p4_s2_d32_layers2_hea...,8d2339746cfe45dd8a7df8ac6d225541,4,2,32,2,4,128.0,0.2,0.0010,0.0005,huber,25,57671,3417.724972,30
3,PatchTST_CalendarAligned_p4_s2_d64_layers2_hea...,46762052992a4c2a91598a29265747d2,4,2,64,2,4,128.0,0.2,0.0005,0.0005,huber,25,131431,3434.974242,15
4,PatchTST_CalendarAligned_p4_s2_d64_layers1_hea...,6dadeba529344daa9b3e85a73e49c6bf,4,2,64,1,4,128.0,0.2,0.0005,0.0005,huber,25,97959,3442.811382,9
5,PatchTST_CalendarAligned_p8_s8_d64_layers1_hea...,5186f361a5d143a3b72abeb18a4ff4cc,8,8,64,1,4,128.0,0.2,0.0005,0.0005,huber,7,52135,3456.816689,27


Saved: /kaggle/working/Walmart/artifacts/patchtst/patchtst_calendar_check_results.csv


### PatchTST-X

In [50]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

runs = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    output_format="pandas",
)

patchtst_x_last39_runs = runs[
    (runs["params.model"] == "PatchTSTXPointForecaster")
    & (runs["params.model_variant"] == "patchtst_x_correction")
    & (runs["params.validation_strategy"] == "last_39_weeks")
    & (runs["tags.mlflow.runName"].astype(str).str.startswith("PatchTSTX_Last39_"))
].copy()

patchtst_x_last39_runs = patchtst_x_last39_runs[
    patchtst_x_last39_runs["metrics.last39_best_valid_wmae"].notna()
].copy()

print("Loaded PatchTST-X last-39 runs from MLflow:", len(patchtst_x_last39_runs))

display(
    patchtst_x_last39_runs[
        [
            "tags.mlflow.runName",
            "run_id",
            "metrics.last39_best_valid_wmae",
            "metrics.last39_best_epoch",
            "metrics.final_correction_scale",
            "params.patch_length",
            "params.stride",
            "params.d_model",
            "params.num_layers",
            "params.attention_heads",
            "params.dim_feedforward",
            "params.embedding_dim",
            "params.exog_hidden_dim",
            "params.dropout",
            "params.learning_rate",
            "params.weight_decay",
            "params.loss_name",
            "params.trainable_parameters",
            "params.num_patches",
        ]
    ].sort_values("metrics.last39_best_valid_wmae")
)

Loaded PatchTST-X last-39 runs from MLflow: 8


,tags.mlflow.runName,run_id,metrics.last39_best_valid_wmae,metrics.last39_best_epoch,metrics.final_correction_scale,params.patch_length,params.stride,params.d_model,params.num_layers,params.attention_heads,params.dim_feedforward,params.embedding_dim,params.exog_hidden_dim,params.dropout,params.learning_rate,params.weight_decay,params.loss_name,params.trainable_parameters,params.num_patches
7,PatchTSTX_Last39_p8_s8_d64_layers2_heads4_exog...,a91c8c59c4d74ba89cf01f108770235d,2237.879817,19.0,0.360847,8,8,64,2,4,None,8,64,0.1,0.001,0.0001,huber,139081,7
5,PatchTSTX_Last39_p4_s4_d64_layers2_heads4_exog...,71ab4d43d7474285b86c7a2c8077947b,2267.005818,12.0,0.339403,4,4,64,2,4,None,8,64,0.1,0.001,0.0001,huber,154185,13
4,PatchTSTX_Last39_p8_s4_d64_layers2_heads4_exog...,e1151a8eb60341ef8abf6529e42c64d5,2277.168950,20.0,0.365842,8,4,64,2,4,None,8,64,0.1,0.001,0.0001,huber,151881,12
6,PatchTSTX_Last39_p4_s2_d64_layers2_heads4_exog...,e94938b9a32f4dbdbaa850f50d952e50,2277.448482,22.0,0.344545,4,2,64,2,4,None,8,64,0.1,0.001,0.0001,huber,184905,25
2,PatchTSTX_Last39_p8_s8_d64_layers2_heads4_exog...,d894a63ad207493bb7b28ae55b3d6d55,2283.560396,19.0,0.304305,8,8,64,2,4,None,8,64,0.2,0.0005,0.0005,huber,139081,7
3,PatchTSTX_Last39_p4_s2_d64_layers2_heads4_exog...,88bcee82e7cd4b0ca0451317c814a00a,2309.348868,11.0,0.270900,4,2,64,2,4,None,8,64,0.2,0.0005,0.0005,huber,184905,25
0,PatchTSTX_Last39_p4_s2_d64_layers2_heads4_exog...,8abea6d492a74c00a9ccf5a61d5cfbfd,2363.205487,25.0,0.328593,4,2,64,2,4,128,8,32,0.2,0.0005,0.0005,huber,138089,25
1,PatchTSTX_Last39_p4_s2_d64_layers1_heads4_exog...,9935ef8a5e3a4fb996a7c54e313d55d2,2368.938770,16.0,0.300828,4,2,64,1,4,128,8,64,0.2,0.0005,0.0005,huber,118409,25


In [51]:
def parse_optional_int(value):
    if pd.isna(value) or value in [None, "None", "nan", ""]:
        return None
    return int(float(value))


patchtst_x_last39_results_df = pd.DataFrame({
    "run_name": patchtst_x_last39_runs["tags.mlflow.runName"],
    "run_id": patchtst_x_last39_runs["run_id"],
    "patch_length": patchtst_x_last39_runs["params.patch_length"].astype(int),
    "stride": patchtst_x_last39_runs["params.stride"].astype(int),
    "d_model": patchtst_x_last39_runs["params.d_model"].astype(int),
    "num_layers": patchtst_x_last39_runs["params.num_layers"].astype(int),
    "attention_heads": patchtst_x_last39_runs["params.attention_heads"].astype(int),
    "dim_feedforward": patchtst_x_last39_runs["params.dim_feedforward"].apply(parse_optional_int),
    "embedding_dim": patchtst_x_last39_runs["params.embedding_dim"].astype(int),
    "exog_hidden_dim": patchtst_x_last39_runs["params.exog_hidden_dim"].astype(int),
    "dropout": patchtst_x_last39_runs["params.dropout"].astype(float),
    "lr": patchtst_x_last39_runs["params.learning_rate"].astype(float),
    "weight_decay": patchtst_x_last39_runs["params.weight_decay"].astype(float),
    "loss_name": patchtst_x_last39_runs["params.loss_name"],
    "use_base_forecast_in_correction": patchtst_x_last39_runs["params.use_base_forecast_in_correction"].astype(str).map({
        "True": True,
        "False": False,
        "true": True,
        "false": False,
    }),
    "correction_scale_init": patchtst_x_last39_runs["params.correction_scale_init"].astype(float),
    "num_patches": patchtst_x_last39_runs["params.num_patches"].astype(int),
    "trainable_parameters": patchtst_x_last39_runs["params.trainable_parameters"].astype(int),
    "last39_best_valid_wmae": patchtst_x_last39_runs["metrics.last39_best_valid_wmae"].astype(float),
    "last39_best_epoch": patchtst_x_last39_runs["metrics.last39_best_epoch"].astype(int),
    "final_correction_scale": patchtst_x_last39_runs["metrics.final_correction_scale"].astype(float),
})

config_subset = [
    "patch_length",
    "stride",
    "d_model",
    "num_layers",
    "attention_heads",
    "dim_feedforward",
    "embedding_dim",
    "exog_hidden_dim",
    "dropout",
    "lr",
    "weight_decay",
    "loss_name",
    "use_base_forecast_in_correction",
    "correction_scale_init",
]

patchtst_x_last39_results_df = (
    patchtst_x_last39_results_df
    .sort_values("last39_best_valid_wmae")
    .drop_duplicates(subset=config_subset, keep="first")
    .reset_index(drop=True)
)

display(patchtst_x_last39_results_df)

x_last39_results_path = ARTIFACT_DIR / "patchtst_x_last39_grid_results_from_mlflow.csv"
patchtst_x_last39_results_df.to_csv(x_last39_results_path, index=False)

print("Saved:", x_last39_results_path)

,run_name,run_id,patch_length,stride,d_model,num_layers,attention_heads,dim_feedforward,embedding_dim,exog_hidden_dim,...,lr,weight_decay,loss_name,use_base_forecast_in_correction,correction_scale_init,num_patches,trainable_parameters,last39_best_valid_wmae,last39_best_epoch,final_correction_scale
0,PatchTSTX_Last39_p8_s8_d64_layers2_heads4_exog...,a91c8c59c4d74ba89cf01f108770235d,8,8,64,2,4,NaN,8,64,...,0.0010,0.0001,huber,True,0.1,7,139081,2237.879817,19,0.360847
1,PatchTSTX_Last39_p4_s4_d64_layers2_heads4_exog...,71ab4d43d7474285b86c7a2c8077947b,4,4,64,2,4,NaN,8,64,...,0.0010,0.0001,huber,True,0.1,13,154185,2267.005818,12,0.339403
2,PatchTSTX_Last39_p8_s4_d64_layers2_heads4_exog...,e1151a8eb60341ef8abf6529e42c64d5,8,4,64,2,4,NaN,8,64,...,0.0010,0.0001,huber,True,0.1,12,151881,2277.168950,20,0.365842
3,PatchTSTX_Last39_p4_s2_d64_layers2_heads4_exog...,e94938b9a32f4dbdbaa850f50d952e50,4,2,64,2,4,NaN,8,64,...,0.0010,0.0001,huber,True,0.1,25,184905,2277.448482,22,0.344545
4,PatchTSTX_Last39_p8_s8_d64_layers2_heads4_exog...,d894a63ad207493bb7b28ae55b3d6d55,8,8,64,2,4,NaN,8,64,...,0.0005,0.0005,huber,True,0.1,7,139081,2283.560396,19,0.304305
5,PatchTSTX_Last39_p4_s2_d64_layers2_heads4_exog...,88bcee82e7cd4b0ca0451317c814a00a,4,2,64,2,4,NaN,8,64,...,0.0005,0.0005,huber,True,0.1,25,184905,2309.348868,11,0.270900
6,PatchTSTX_Last39_p4_s2_d64_layers2_heads4_exog...,8abea6d492a74c00a9ccf5a61d5cfbfd,4,2,64,2,4,128.0,8,32,...,0.0005,0.0005,huber,True,0.1,25,138089,2363.205487,25,0.328593
7,PatchTSTX_Last39_p4_s2_d64_layers1_heads4_exog...,9935ef8a5e3a4fb996a7c54e313d55d2,4,2,64,1,4,128.0,8,64,...,0.0005,0.0005,huber,True,0.1,25,118409,2368.938770,16,0.300828


Saved: /kaggle/working/Walmart/artifacts/patchtst/patchtst_x_last39_grid_results_from_mlflow.csv


In [52]:
top_configs = (
    patchtst_x_last39_results_df
    .sort_values("last39_best_valid_wmae")
    .head(3)
    .copy()
)

diverse_configs = []

# best p8/s8 setup
diverse_configs.append(
    patchtst_x_last39_results_df[
        (patchtst_x_last39_results_df["patch_length"] == 8)
        & (patchtst_x_last39_results_df["stride"] == 8)
    ].sort_values("last39_best_valid_wmae").head(1)
)

# best p4/s2 setup
diverse_configs.append(
    patchtst_x_last39_results_df[
        (patchtst_x_last39_results_df["patch_length"] == 4)
        & (patchtst_x_last39_results_df["stride"] == 2)
    ].sort_values("last39_best_valid_wmae").head(1)
)

# best regularized setup
diverse_configs.append(
    patchtst_x_last39_results_df[
        patchtst_x_last39_results_df["dropout"] >= 0.2
    ].sort_values("last39_best_valid_wmae").head(1)
)

# best smaller correction head if present
diverse_configs.append(
    patchtst_x_last39_results_df[
        patchtst_x_last39_results_df["exog_hidden_dim"] == 32
    ].sort_values("last39_best_valid_wmae").head(1)
)

patchtst_x_calendar_candidate_configs = pd.concat(
    [top_configs] + diverse_configs,
    axis=0,
).drop_duplicates(
    subset=[
        "patch_length",
        "stride",
        "d_model",
        "num_layers",
        "attention_heads",
        "dim_feedforward",
        "embedding_dim",
        "exog_hidden_dim",
        "dropout",
        "lr",
        "weight_decay",
        "loss_name",
        "use_base_forecast_in_correction",
        "correction_scale_init",
    ]
).sort_values(
    "last39_best_valid_wmae"
).reset_index(drop=True)

display(patchtst_x_calendar_candidate_configs)
print("PatchTST-X calendar candidate configs:", len(patchtst_x_calendar_candidate_configs))

,run_name,run_id,patch_length,stride,d_model,num_layers,attention_heads,dim_feedforward,embedding_dim,exog_hidden_dim,...,lr,weight_decay,loss_name,use_base_forecast_in_correction,correction_scale_init,num_patches,trainable_parameters,last39_best_valid_wmae,last39_best_epoch,final_correction_scale
0,PatchTSTX_Last39_p8_s8_d64_layers2_heads4_exog...,a91c8c59c4d74ba89cf01f108770235d,8,8,64,2,4,NaN,8,64,...,0.0010,0.0001,huber,True,0.1,7,139081,2237.879817,19,0.360847
1,PatchTSTX_Last39_p4_s4_d64_layers2_heads4_exog...,71ab4d43d7474285b86c7a2c8077947b,4,4,64,2,4,NaN,8,64,...,0.0010,0.0001,huber,True,0.1,13,154185,2267.005818,12,0.339403
2,PatchTSTX_Last39_p8_s4_d64_layers2_heads4_exog...,e1151a8eb60341ef8abf6529e42c64d5,8,4,64,2,4,NaN,8,64,...,0.0010,0.0001,huber,True,0.1,12,151881,2277.168950,20,0.365842
3,PatchTSTX_Last39_p4_s2_d64_layers2_heads4_exog...,e94938b9a32f4dbdbaa850f50d952e50,4,2,64,2,4,NaN,8,64,...,0.0010,0.0001,huber,True,0.1,25,184905,2277.448482,22,0.344545
4,PatchTSTX_Last39_p8_s8_d64_layers2_heads4_exog...,d894a63ad207493bb7b28ae55b3d6d55,8,8,64,2,4,NaN,8,64,...,0.0005,0.0005,huber,True,0.1,7,139081,2283.560396,19,0.304305
5,PatchTSTX_Last39_p4_s2_d64_layers2_heads4_exog...,8abea6d492a74c00a9ccf5a61d5cfbfd,4,2,64,2,4,128.0,8,32,...,0.0005,0.0005,huber,True,0.1,25,138089,2363.205487,25,0.328593


PatchTST-X calendar candidate configs: 6


In [53]:
PATCHTST_X_CALENDAR_EPOCHS = 50

patchtst_x_calendar_results = []

START_FROM_CONFIG = 1

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print("PatchTST-X calendar candidate configs:", len(patchtst_x_calendar_candidate_configs))
print("Epochs per config:", PATCHTST_X_CALENDAR_EPOCHS)

for i, row in patchtst_x_calendar_candidate_configs.iterrows():
    run_number = i + 1

    if run_number < START_FROM_CONFIG:
        continue

    config = {
        "patch_length": int(row["patch_length"]),
        "stride": int(row["stride"]),
        "d_model": int(row["d_model"]),
        "num_layers": int(row["num_layers"]),
        "attention_heads": int(row["attention_heads"]),
        "dim_feedforward": parse_optional_int(row["dim_feedforward"]),
        "embedding_dim": int(row["embedding_dim"]),
        "exog_hidden_dim": int(row["exog_hidden_dim"]),
        "dropout": float(row["dropout"]),
        "lr": float(row["lr"]),
        "weight_decay": float(row["weight_decay"]),
        "loss_name": row["loss_name"],
        "use_base_forecast_in_correction": bool(row["use_base_forecast_in_correction"]),
        "correction_scale_init": float(row["correction_scale_init"]),
    }

    run_name = (
        f"PatchTSTX_CalendarAligned_"
        f"p{config['patch_length']}_"
        f"s{config['stride']}_"
        f"d{config['d_model']}_"
        f"layers{config['num_layers']}_"
        f"heads{config['attention_heads']}_"
        f"exog{config['exog_hidden_dim']}_"
        f"drop{safe_float_name(config['dropout'])}_"
        f"lr{safe_float_name(config['lr'])}_"
        f"wd{safe_float_name(config['weight_decay'])}_"
        f"{config['loss_name']}"
    )

    print("=" * 100)
    print(f"Calendar run {run_number}/{len(patchtst_x_calendar_candidate_configs)}: {run_name}")
    print(config)

    set_seed(SEED + 300 + run_number)

    model = PatchTSTXPointForecaster(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        static_cat_cardinalities=patchtst_calendar_x_data["static_cat_cardinalities"],
        num_known_reals=patchtst_calendar_x_data["num_known_reals"],
        num_static_reals=patchtst_calendar_x_data["num_static_reals"],
        patch_length=config["patch_length"],
        stride=config["stride"],
        d_model=config["d_model"],
        num_layers=config["num_layers"],
        attention_heads=config["attention_heads"],
        dim_feedforward=config["dim_feedforward"],
        embedding_dim=config["embedding_dim"],
        exog_hidden_dim=config["exog_hidden_dim"],
        dropout=config["dropout"],
        use_base_forecast_in_correction=config["use_base_forecast_in_correction"],
        correction_scale_init=config["correction_scale_init"],
    ).to(DEVICE)

    n_params = count_parameters(model)

    print("Trainable parameters:", n_params)
    print("Number of patches:", model.target_backbone.num_patches)
    print("Initial correction scale:", float(model.correction_scale.detach().cpu()))

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_loss_fn(config["loss_name"])

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id

        mlflow.log_params({
            "model": "PatchTSTXPointForecaster",
            "model_variant": "patchtst_x_correction",
            "feature_set": "stable_no_markdowns_exogenous_correction",
            "validation_strategy": "calendar_aligned_39_weeks",
            "calendar_valid_start": "2011-11-04",
            "calendar_valid_end": "2012-07-27",
            "context_length": CONTEXT_LENGTH,
            "prediction_length": PREDICTION_LENGTH,
            "patch_length": config["patch_length"],
            "stride": config["stride"],
            "num_patches": model.target_backbone.num_patches,
            "d_model": config["d_model"],
            "num_layers": config["num_layers"],
            "attention_heads": config["attention_heads"],
            "dim_feedforward": config["dim_feedforward"],
            "embedding_dim": config["embedding_dim"],
            "exog_hidden_dim": config["exog_hidden_dim"],
            "dropout": config["dropout"],
            "learning_rate": config["lr"],
            "weight_decay": config["weight_decay"],
            "loss_name": config["loss_name"],
            "batch_size": BATCH_SIZE,
            "epochs": PATCHTST_X_CALENDAR_EPOCHS,
            "optimizer": "AdamW",
            "train_loader_type": "FastTensorDataLoader",
            "train_windows": len(patchtst_calendar_x_data["train_dataset"]),
            "valid_series": len(patchtst_calendar_x_data["valid_dataset"]),
            "num_known_reals": patchtst_calendar_x_data["num_known_reals"],
            "num_static_reals": patchtst_calendar_x_data["num_static_reals"],
            "static_cat_cardinalities": str(patchtst_calendar_x_data["static_cat_cardinalities"]),
            "use_base_forecast_in_correction": config["use_base_forecast_in_correction"],
            "correction_scale_init": config["correction_scale_init"],
            "trainable_parameters": n_params,
            "source_last39_run_id": row["run_id"],
            "source_last39_best_valid_wmae": float(row["last39_best_valid_wmae"]),
            "source_last39_best_epoch": int(row["last39_best_epoch"]),
            "source_last39_final_correction_scale": float(row["final_correction_scale"]),
        })

        result = fit_model(
            model=model,
            train_loader=patchtst_calendar_x_data["train_loader"],
            valid_loader=patchtst_calendar_x_data["valid_loader"],
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=DEVICE,
            holiday_feature_idx=patchtst_calendar_x_data["holiday_feature_idx"],
            epochs=PATCHTST_X_CALENDAR_EPOCHS,
            metric_prefix="calendar",
        )

        best_wmae = result["best_valid_wmae"]
        best_epoch = result["best_epoch"]

        mlflow.log_metric("calendar_best_valid_wmae", best_wmae)
        mlflow.log_metric("calendar_best_epoch", best_epoch)
        mlflow.log_metric("final_correction_scale", float(model.correction_scale.detach().cpu()))

    patchtst_x_calendar_results.append({
        "run_name": run_name,
        "run_id": run_id,
        **config,
        "num_patches": model.target_backbone.num_patches,
        "trainable_parameters": n_params,
        "source_last39_run_id": row["run_id"],
        "source_last39_best_valid_wmae": float(row["last39_best_valid_wmae"]),
        "source_last39_best_epoch": int(row["last39_best_epoch"]),
        "source_last39_final_correction_scale": float(row["final_correction_scale"]),
        "calendar_best_valid_wmae": best_wmae,
        "calendar_best_epoch": best_epoch,
        "final_correction_scale": float(model.correction_scale.detach().cpu()),
    })

    print(f"Best calendar WMAE: {best_wmae:.2f}")
    print(f"Best epoch: {best_epoch}")
    print("Final correction scale:", float(model.correction_scale.detach().cpu()))

    del model
    del optimizer
    del loss_fn

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

patchtst_x_calendar_results_df = (
    pd.DataFrame(patchtst_x_calendar_results)
    .sort_values("calendar_best_valid_wmae")
    .reset_index(drop=True)
)

display(patchtst_x_calendar_results_df)

x_calendar_results_path = ARTIFACT_DIR / "patchtst_x_calendar_check_results.csv"
patchtst_x_calendar_results_df.to_csv(x_calendar_results_path, index=False)

print("Saved:", x_calendar_results_path)

PatchTST-X calendar candidate configs: 6
Epochs per config: 50
Calendar run 1/6: PatchTSTX_CalendarAligned_p8_s8_d64_layers2_heads4_exog64_drop0p1_lr0p001_wd0p0001_huber
{'patch_length': 8, 'stride': 8, 'd_model': 64, 'num_layers': 2, 'attention_heads': 4, 'dim_feedforward': None, 'embedding_dim': 8, 'exog_hidden_dim': 64, 'dropout': 0.1, 'lr': 0.001, 'weight_decay': 0.0001, 'loss_name': 'huber', 'use_base_forecast_in_correction': True, 'correction_scale_init': 0.1}
Trainable parameters: 139081
Number of patches: 7
Initial correction scale: 0.10000000149011612
Epoch 001 | train_loss=0.29770 | valid_loss=0.59707 | valid_wmae=3751.30 | valid_mae=3322.29
Epoch 002 | train_loss=0.21469 | valid_loss=0.56920 | valid_wmae=3524.18 | valid_mae=3123.48
Epoch 003 | train_loss=0.19487 | valid_loss=0.57391 | valid_wmae=3525.01 | valid_mae=3122.32
Epoch 004 | train_loss=0.18440 | valid_loss=0.56883 | valid_wmae=3471.07 | valid_mae=3085.78
Epoch 005 | train_loss=0.17622 | valid_loss=0.56527 | valid_w

,run_name,run_id,patch_length,stride,d_model,num_layers,attention_heads,dim_feedforward,embedding_dim,exog_hidden_dim,...,correction_scale_init,num_patches,trainable_parameters,source_last39_run_id,source_last39_best_valid_wmae,source_last39_best_epoch,source_last39_final_correction_scale,calendar_best_valid_wmae,calendar_best_epoch,final_correction_scale
0,PatchTSTX_CalendarAligned_p4_s4_d64_layers2_he...,d559d9374e78459c9b6342a463dfd143,4,4,64,2,4,NaN,8,64,...,0.1,13,154185,71ab4d43d7474285b86c7a2c8077947b,2267.005818,12,0.339403,3382.656467,5,0.138535
1,PatchTSTX_CalendarAligned_p8_s8_d64_layers2_he...,076fcc4a33a24c4eab0d3f20908f06fe,8,8,64,2,4,NaN,8,64,...,0.1,7,139081,a91c8c59c4d74ba89cf01f108770235d,2237.879817,19,0.360847,3410.877951,9,0.148846
2,PatchTSTX_CalendarAligned_p8_s8_d64_layers2_he...,ca50e9e5ab664780a41ce0c5fdfc1bb8,8,8,64,2,4,NaN,8,64,...,0.1,7,139081,d894a63ad207493bb7b28ae55b3d6d55,2283.560396,19,0.304305,3435.569261,29,0.200924
3,PatchTSTX_CalendarAligned_p4_s2_d64_layers2_he...,08f52a2f5bc547c18a70a67cdc8c5cf7,4,2,64,2,4,NaN,8,64,...,0.1,25,184905,e94938b9a32f4dbdbaa850f50d952e50,2277.448482,22,0.344545,3440.409850,3,0.115876
4,PatchTSTX_CalendarAligned_p4_s2_d64_layers2_he...,1e2e5a6d394f4cdaa85c517e0e221408,4,2,64,2,4,128.0,8,32,...,0.1,25,138089,8abea6d492a74c00a9ccf5a61d5cfbfd,2363.205487,25,0.328593,3442.495109,22,0.193838
5,PatchTSTX_CalendarAligned_p8_s4_d64_layers2_he...,1af0c7a1c9cb4ed1bf35495785680a9e,8,4,64,2,4,NaN,8,64,...,0.1,12,151881,e1151a8eb60341ef8abf6529e42c64d5,2277.168950,20,0.365842,3448.237443,6,0.141678


Saved: /kaggle/working/Walmart/artifacts/patchtst/patchtst_x_calendar_check_results.csv


## Kaggle Check


In [54]:
# full train preprocessing for final test inference

final_base_preprocessor = WalmartBasePreprocessor()
final_base_preprocessor.fit(stores, features)

full_train_base = final_base_preprocessor.transform(train)

final_neural_preprocessor = WalmartNeuralPreprocessor()
final_neural_preprocessor.fit(full_train_base)

full_train_panel = final_neural_preprocessor.transform(full_train_base)

final_dataset_cols = final_neural_preprocessor.get_dataset_columns()

print("Full train panel shape:", full_train_panel.shape)
print("Full train date range:", full_train_panel["Date"].min(), "->", full_train_panel["Date"].max())
print("Full train unique dates:", full_train_panel["Date"].nunique())
print(final_dataset_cols)

Full train panel shape: (421570, 66)
Full train date range: 2010-02-05 00:00:00 -> 2012-10-26 00:00:00
Full train unique dates: 143
{'target_col': 'Weekly_Sales_scaled', 'series_col': 'series_id', 'static_cat_cols': ['Store_id', 'Dept_id', 'Type_id'], 'static_real_cols': ['Size_scaled'], 'known_future_real_cols': ['Temperature_scaled', 'Fuel_Price_scaled', 'CPI_scaled', 'Unemployment_scaled', 'MarkDown1_scaled', 'MarkDown2_scaled', 'MarkDown3_scaled', 'MarkDown4_scaled', 'MarkDown5_scaled', 'total_markdown_scaled', 'abs_total_markdown_scaled', 'positive_markdown_sum_scaled', 'negative_markdown_sum_scaled', 'markdown_missing_count_scaled', 'Week_sin_scaled', 'Week_cos_scaled', 'IsHoliday', 'IsSuperBowl', 'IsLaborDay', 'IsThanksgiving', 'IsChristmas', 'has_markdown_signal', 'markdown_available_period', 'MarkDown1_was_missing', 'MarkDown2_was_missing', 'MarkDown3_was_missing', 'MarkDown4_was_missing', 'MarkDown5_was_missing']}


In [55]:
test_dates = pd.DataFrame({
    "Date": sorted(pd.to_datetime(test["Date"]).unique())
})

test_pairs = test[["Store", "Dept"]].drop_duplicates().reset_index(drop=True)

# cross join: every test pair gets all 39 test dates (later we will merge IsHoliday from original test )
test_grid = test_pairs.merge(test_dates, how="cross")

# IsHoliday depends on Date so map it from original test
date_holiday = (
    test[["Date", "IsHoliday"]]
    .copy()
    .assign(Date=lambda x: pd.to_datetime(x["Date"]))
    .drop_duplicates()
)

assert date_holiday["Date"].nunique() == len(date_holiday)

test_grid = test_grid.merge(date_holiday, on="Date", how="left")

print("Original test rows:", len(test))
print("Full test grid rows:", len(test_grid))
print("Test pairs:", len(test_pairs))
print("Test dates:", len(test_dates))

assert test_grid["IsHoliday"].notna().all()
assert len(test_dates) == PREDICTION_LENGTH

Original test rows: 115064
Full test grid rows: 123591
Test pairs: 3169
Test dates: 39


In [56]:
full_test_grid_base = final_base_preprocessor.transform(test_grid)
full_test_grid_panel = final_neural_preprocessor.transform(full_test_grid_base)

print("Full test grid panel shape:", full_test_grid_panel.shape)
print("Full test grid date range:", full_test_grid_panel["Date"].min(), "->", full_test_grid_panel["Date"].max())
print("Full test grid unique dates:", full_test_grid_panel["Date"].nunique())

test_group_sizes = full_test_grid_panel.groupby("series_id").size()
assert (test_group_sizes == PREDICTION_LENGTH).all()

print("All test Store-Dept groups have 39 rows.")

Full test grid panel shape: (123591, 65)
Full test grid date range: 2012-11-02 00:00:00 -> 2013-07-26 00:00:00
Full test grid unique dates: 39
All test Store-Dept groups have 39 rows.


In [88]:
BEST_PATCHTST_TARGET_FINAL_CONFIG_LAST39 = {
    "patch_length": 8,
    "stride": 8,
    "d_model": 64,
    "num_layers": 2,
    "attention_heads": 4,
    "dim_feedforward": None,
    "dropout": 0.1,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "loss_name": "huber",
    
    "final_epochs": 24,
}

BEST_PATCHTST_TARGET_FINAL_CONFIG_CALENDAR = {
    "patch_length": 4,
    "stride": 2,
    "d_model": 64,
    "num_layers": 2,
    "attention_heads": 4,
    "dim_feedforward": None,
    "dropout": 0.1,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "loss_name": "huber",
    
    "final_epochs": 3,
}

BEST_PATCHTST_X_FINAL_CONFIG_LAST39 = {
    "patch_length": 8,
    "stride": 8,
    "d_model": 64,
    "num_layers": 2,
    "attention_heads": 4,
    "dim_feedforward": None,
    "embedding_dim": 8,
    "exog_hidden_dim": 64,
    "dropout": 0.1,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "loss_name": "huber",
    "use_base_forecast_in_correction": True,
    "correction_scale_init": 0.1,
    
    "final_epochs": 19,
}

BEST_PATCHTST_X_FINAL_CONFIG_CALENDAR = {
    "patch_length": 4,
    "stride": 4,
    "d_model": 64,
    "num_layers": 2,
    "attention_heads": 4,
    "dim_feedforward": None,
    "embedding_dim": 8,
    "exog_hidden_dim": 64,
    "dropout": 0.1,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "loss_name": "huber",
    "use_base_forecast_in_correction": True,
    "correction_scale_init": 0.1,
    
    "final_epochs": 5,
}

BEST_PATCHTST_TARGET_FINAL_CONFIG = BEST_PATCHTST_TARGET_FINAL_CONFIG_CALENDAR
BEST_PATCHTST_X_FINAL_CONFIG = BEST_PATCHTST_X_FINAL_CONFIG_CALENDAR


final_patchtst_target_cols = {
    "target_col": final_dataset_cols["target_col"],
    "series_col": final_dataset_cols["series_col"],
    "static_cat_cols": [],
    "static_real_cols": [],
    "known_future_real_cols": ["IsHoliday"],
}

FINAL_PATCHTST_TARGET_BATCH_SIZE = 512

final_patchtst_target_train_dataset = WalmartPrecomputedTrainingWindowDataset(
    full_train_panel,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    target_col=final_patchtst_target_cols["target_col"],
    series_col=final_patchtst_target_cols["series_col"],
    static_cat_cols=final_patchtst_target_cols["static_cat_cols"],
    static_real_cols=final_patchtst_target_cols["static_real_cols"],
    known_future_real_cols=final_patchtst_target_cols["known_future_real_cols"],
)

final_patchtst_target_test_dataset = WalmartPrecomputedForecastWindowDataset(
    history_df=full_train_panel,
    future_df=full_test_grid_panel,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    target_col=final_patchtst_target_cols["target_col"],
    series_col=final_patchtst_target_cols["series_col"],
    static_cat_cols=final_patchtst_target_cols["static_cat_cols"],
    static_real_cols=final_patchtst_target_cols["static_real_cols"],
    known_future_real_cols=final_patchtst_target_cols["known_future_real_cols"],
)

final_patchtst_target_train_loader = FastTensorDataLoader(
    final_patchtst_target_train_dataset.tensors,
    batch_size=FINAL_PATCHTST_TARGET_BATCH_SIZE,
    shuffle=True,
)

final_patchtst_target_test_loader = FastTensorDataLoader(
    final_patchtst_target_test_dataset.tensors,
    batch_size=FINAL_PATCHTST_TARGET_BATCH_SIZE,
    shuffle=False,
)

final_patchtst_target_holiday_feature_idx = final_patchtst_target_cols["known_future_real_cols"].index("IsHoliday")

print("Final PatchTST target-only train windows:", len(final_patchtst_target_train_dataset))
print("Final PatchTST target-only test forecast series:", len(final_patchtst_target_test_dataset))
print("Final PatchTST target-only train batches:", len(final_patchtst_target_train_loader))
print("Final PatchTST target-only test batches:", len(final_patchtst_target_test_loader))
print("Holiday idx:", final_patchtst_target_holiday_feature_idx)

assert final_patchtst_target_test_dataset.tensors["past_target"].shape[1] == CONTEXT_LENGTH
assert final_patchtst_target_test_dataset.tensors["future_known_reals"].shape[1] == PREDICTION_LENGTH

Final PatchTST target-only train windows: 149069
Final PatchTST target-only test forecast series: 3169
Final PatchTST target-only train batches: 292
Final PatchTST target-only test batches: 7
Holiday idx: 0


In [78]:
def make_final_patchtst_target_loss(config: dict, holiday_feature_idx: int):
    if config["loss_name"] == "weighted_huber":
        return WeightedHuberLoss(
            holiday_feature_idx=holiday_feature_idx,
            holiday_weight=config["holiday_weight"],
            delta=config["delta"],
        )

    return make_loss_fn(config["loss_name"])


def train_final_patchtst_target(
    config: dict,
    train_loader,
    device,
):
    model = PatchTSTPointForecaster(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        patch_length=config["patch_length"],
        stride=config["stride"],
        d_model=config["d_model"],
        num_layers=config["num_layers"],
        attention_heads=config["attention_heads"],
        dim_feedforward=config["dim_feedforward"],
        dropout=config["dropout"],
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_final_patchtst_target_loss(
        config=config,
        holiday_feature_idx=final_patchtst_target_holiday_feature_idx,
    )

    epochs = config["final_epochs"]

    print("Trainable parameters:", count_parameters(model))
    print("Number of patches:", model.num_patches)
    print("Final training epochs:", epochs)

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=device,
        )

        print(f"Epoch {epoch:03d} | final_train_loss={train_loss:.5f}")

    return model


@torch.no_grad()
def predict_patchtst_target(model, loader, device):
    model.eval()

    all_preds = []

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        preds_scaled = forward_model(model, batch)

        preds_original = (
            preds_scaled * batch["target_std"].unsqueeze(1)
            + batch["target_mean"].unsqueeze(1)
        )

        all_preds.append(preds_original.detach().cpu().numpy())

    return np.concatenate(all_preds, axis=0)

In [93]:
def make_final_patchtst_x_loss(config: dict, holiday_feature_idx: int):
    if config["loss_name"] == "weighted_huber":
        return WeightedHuberLoss(
            holiday_feature_idx=holiday_feature_idx,
            holiday_weight=config["holiday_weight"],
            delta=config["delta"],
        )

    return make_loss_fn(config["loss_name"])


def train_final_patchtst_x(
    config: dict,
    train_loader,
    device,
):
    model = PatchTSTXPointForecaster(
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        static_cat_cardinalities=final_patchtst_x_static_cat_cardinalities,
        num_known_reals=final_patchtst_x_num_known_reals,
        num_static_reals=final_patchtst_x_num_static_reals,
        patch_length=config["patch_length"],
        stride=config["stride"],
        d_model=config["d_model"],
        num_layers=config["num_layers"],
        attention_heads=config["attention_heads"],
        dim_feedforward=config["dim_feedforward"],
        embedding_dim=config["embedding_dim"],
        exog_hidden_dim=config["exog_hidden_dim"],
        dropout=config["dropout"],
        use_base_forecast_in_correction=config["use_base_forecast_in_correction"],
        correction_scale_init=config["correction_scale_init"],
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    loss_fn = make_final_patchtst_x_loss(
        config=config,
        holiday_feature_idx=final_patchtst_x_holiday_feature_idx,
    )

    epochs = config["final_epochs"]

    print("Trainable parameters:", count_parameters(model))
    print("Number of patches:", model.target_backbone.num_patches)
    print("Initial correction scale:", float(model.correction_scale.detach().cpu()))
    print("Final training epochs:", epochs)

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=device,
        )

        print(f"Epoch {epoch:03d} | final_train_loss={train_loss:.5f}")

    print("Final correction scale:", float(model.correction_scale.detach().cpu()))

    return model


@torch.no_grad()
def predict_patchtst_x(model, loader, device):
    model.eval()

    all_preds = []

    for batch in loader:
        batch = move_batch_to_device(batch, device)

        preds_scaled = forward_model(model, batch)

        preds_original = (
            preds_scaled * batch["target_std"].unsqueeze(1)
            + batch["target_mean"].unsqueeze(1)
        )

        all_preds.append(preds_original.detach().cpu().numpy())

    return np.concatenate(all_preds, axis=0)

In [79]:
set_seed(SEED + 940)

final_patchtst_target_model = train_final_patchtst_target(
    BEST_PATCHTST_TARGET_FINAL_CONFIG,
    final_patchtst_target_train_loader,
    DEVICE,
)

patchtst_target_test_preds_matrix = predict_patchtst_target(
    final_patchtst_target_model,
    final_patchtst_target_test_loader,
    DEVICE,
)

print("PatchTST target-only test prediction matrix shape:", patchtst_target_test_preds_matrix.shape)

assert patchtst_target_test_preds_matrix.shape == (
    len(final_patchtst_target_test_dataset),
    PREDICTION_LENGTH,
)

Trainable parameters: 164455
Number of patches: 25
Final training epochs: 3
Epoch 001 | final_train_loss=0.23522
Epoch 002 | final_train_loss=0.20550
Epoch 003 | final_train_loss=0.19947
PatchTST target-only test prediction matrix shape: (3169, 39)


In [90]:
set_seed(SEED + 950)

final_patchtst_x_model = train_final_patchtst_x(
    BEST_PATCHTST_X_FINAL_CONFIG,
    final_patchtst_x_train_loader,
    DEVICE,
)

patchtst_x_test_preds_matrix = predict_patchtst_x(
    final_patchtst_x_model,
    final_patchtst_x_test_loader,
    DEVICE,
)

print("PatchTST-X test prediction matrix shape:", patchtst_x_test_preds_matrix.shape)

assert patchtst_x_test_preds_matrix.shape == (
    len(final_patchtst_x_test_dataset),
    PREDICTION_LENGTH,
)

Trainable parameters: 154185
Number of patches: 13
Initial correction scale: 0.10000000149011612
Final training epochs: 5
Epoch 001 | final_train_loss=0.22909
Epoch 002 | final_train_loss=0.18660
Epoch 003 | final_train_loss=0.17619
Epoch 004 | final_train_loss=0.17015
Epoch 005 | final_train_loss=0.16563
Final correction scale: 0.3084024488925934
PatchTST-X test prediction matrix shape: (3169, 39)


In [80]:
test_index_df = final_patchtst_target_test_dataset.get_future_index().reset_index(drop=True).copy()

patchtst_target_test_preds_flat = patchtst_target_test_preds_matrix.reshape(-1)

assert len(test_index_df) == len(patchtst_target_test_preds_flat)

full_patchtst_target_pred_df = test_index_df.copy()
full_patchtst_target_pred_df["Weekly_Sales"] = patchtst_target_test_preds_flat

negative_before_clip = (full_patchtst_target_pred_df["Weekly_Sales"] < 0).sum()
print("Negative predictions before clipping:", negative_before_clip)

full_patchtst_target_pred_df["Weekly_Sales"] = full_patchtst_target_pred_df["Weekly_Sales"].clip(lower=0)

print(full_patchtst_target_pred_df.head())
print(full_patchtst_target_pred_df["Weekly_Sales"].describe())

Negative predictions before clipping: 2346
   Store  Dept       Date  Weekly_Sales
0     10     1 2012-11-02  62185.851562
1     10     1 2012-11-09  45570.496094
2     10     1 2012-11-16  39892.742188
3     10     1 2012-11-23  47100.414062
4     10     1 2012-11-30  48616.015625
count    123591.000000
mean      14974.501953
std       22224.119141
min           0.000000
25%        1366.381104
50%        6577.826172
75%       18898.401367
max      512236.187500
Name: Weekly_Sales, dtype: float64


In [91]:
test_index_df = final_patchtst_x_test_dataset.get_future_index().reset_index(drop=True).copy()

patchtst_x_test_preds_flat = patchtst_x_test_preds_matrix.reshape(-1)

assert len(test_index_df) == len(patchtst_x_test_preds_flat)

full_patchtst_x_pred_df = test_index_df.copy()
full_patchtst_x_pred_df["Weekly_Sales"] = patchtst_x_test_preds_flat

negative_before_clip = (full_patchtst_x_pred_df["Weekly_Sales"] < 0).sum()
print("Negative predictions before clipping:", negative_before_clip)

# postprocessing: clip negative sales predictions to zero
full_patchtst_x_pred_df["Weekly_Sales"] = full_patchtst_x_pred_df["Weekly_Sales"].clip(lower=0)

print(full_patchtst_x_pred_df.head())
print(full_patchtst_x_pred_df["Weekly_Sales"].describe())

Negative predictions before clipping: 2119
   Store  Dept       Date  Weekly_Sales
0     10     1 2012-11-02  76030.140625
1     10     1 2012-11-09  43283.609375
2     10     1 2012-11-16  31241.289062
3     10     1 2012-11-23  38176.214844
4     10     1 2012-11-30  47449.964844
count    123591.000000
mean      15218.492188
std       22881.412109
min           0.000000
25%        1373.473022
50%        6612.085938
75%       19218.063477
max      625543.750000
Name: Weekly_Sales, dtype: float64


In [81]:
test_keys = test[["Store", "Dept", "Date"]].copy()
test_keys["Date"] = pd.to_datetime(test_keys["Date"])

patchtst_target_submission_df = test_keys.merge(
    full_patchtst_target_pred_df,
    on=["Store", "Dept", "Date"],
    how="left",
)

assert len(patchtst_target_submission_df) == len(test)
assert patchtst_target_submission_df["Weekly_Sales"].notna().all()

patchtst_target_submission_df["Id"] = (
    patchtst_target_submission_df["Store"].astype(str)
    + "_"
    + patchtst_target_submission_df["Dept"].astype(str)
    + "_"
    + patchtst_target_submission_df["Date"].dt.strftime("%Y-%m-%d")
)

patchtst_target_submission_df = patchtst_target_submission_df[["Id", "Weekly_Sales"]]

print(patchtst_target_submission_df.head())
print(patchtst_target_submission_df.shape)

submission_name = (
    "submission_patchtst_target_"
    f"p{BEST_PATCHTST_TARGET_FINAL_CONFIG['patch_length']}_"
    f"s{BEST_PATCHTST_TARGET_FINAL_CONFIG['stride']}_"
    f"d{BEST_PATCHTST_TARGET_FINAL_CONFIG['d_model']}_"
    f"layers{BEST_PATCHTST_TARGET_FINAL_CONFIG['num_layers']}_"
    f"drop{BEST_PATCHTST_TARGET_FINAL_CONFIG['dropout']}_"
    f"{BEST_PATCHTST_TARGET_FINAL_CONFIG['loss_name']}_"
    f"epochs{BEST_PATCHTST_TARGET_FINAL_CONFIG['final_epochs']}.csv"
)

submission_path = repo_root / submission_name

patchtst_target_submission_df.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)

               Id  Weekly_Sales
0  1_1_2012-11-02  30804.335938
1  1_1_2012-11-09  23396.466797
2  1_1_2012-11-16  21942.496094
3  1_1_2012-11-23  23671.072266
4  1_1_2012-11-30  26262.515625
(115064, 2)
Saved submission to: /kaggle/working/Walmart/submission_patchtst_target_p4_s2_d64_layers2_drop0.1_huber_epochs3.csv


In [92]:
test_keys = test[["Store", "Dept", "Date"]].copy()
test_keys["Date"] = pd.to_datetime(test_keys["Date"])

patchtst_x_submission_df = test_keys.merge(
    full_patchtst_x_pred_df,
    on=["Store", "Dept", "Date"],
    how="left",
)

assert len(patchtst_x_submission_df) == len(test)
assert patchtst_x_submission_df["Weekly_Sales"].notna().all()

patchtst_x_submission_df["Id"] = (
    patchtst_x_submission_df["Store"].astype(str)
    + "_"
    + patchtst_x_submission_df["Dept"].astype(str)
    + "_"
    + patchtst_x_submission_df["Date"].dt.strftime("%Y-%m-%d")
)

patchtst_x_submission_df = patchtst_x_submission_df[["Id", "Weekly_Sales"]]

print(patchtst_x_submission_df.head())
print(patchtst_x_submission_df.shape)

submission_name = (
    "submission_patchtst_x_"
    f"p{BEST_PATCHTST_X_FINAL_CONFIG['patch_length']}_"
    f"s{BEST_PATCHTST_X_FINAL_CONFIG['stride']}_"
    f"d{BEST_PATCHTST_X_FINAL_CONFIG['d_model']}_"
    f"layers{BEST_PATCHTST_X_FINAL_CONFIG['num_layers']}_"
    f"exog{BEST_PATCHTST_X_FINAL_CONFIG['exog_hidden_dim']}_"
    f"drop{BEST_PATCHTST_X_FINAL_CONFIG['dropout']}_"
    f"{BEST_PATCHTST_X_FINAL_CONFIG['loss_name']}_"
    f"epochs{BEST_PATCHTST_X_FINAL_CONFIG['final_epochs']}.csv"
)

submission_path = repo_root / submission_name

patchtst_x_submission_df.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)

               Id  Weekly_Sales
0  1_1_2012-11-02  38532.507812
1  1_1_2012-11-09  23305.865234
2  1_1_2012-11-16  18308.318359
3  1_1_2012-11-23  19512.677734
4  1_1_2012-11-30  26163.851562
(115064, 2)
Saved submission to: /kaggle/working/Walmart/submission_patchtst_x_p4_s4_d64_layers2_exog64_drop0.1_huber_epochs5.csv


PatchTST-X model seems to be a clear winner in both validation splits, both calendar winner `patchtst_x_p4_s4_d64_layers2_exog64_drop0.1_huber_epochs5` and last_39 winner `patchtst_x_p8_s8_d64_layers2_exog64_drop0.1_huber_epochs19` got better test scores than either target-only model.